# DataWorks Challenge 2026: history matching and production forecasting

Reservoir parameters → production proxy → history matching → ten optimized field forecasts
through January 2028. Original simulator cases provide a separate benchmark. Optimized
realizations are proxy predictions, not new simulator runs. Production truth after
January 2008 is not supplied.

**Local setup:** use Python 3.12, install `requirements-lock.txt` for the tested Windows
environment (or `requirements.txt` elsewhere), select that environment as the notebook
kernel, and run all cells from the repository root. Put the official workbooks in
`TestCases/`. Full setup, artifact descriptions and evaluation provenance are in `README.md`
and `docs/METHODOLOGY.md`.

**Colab:** keep the notebook, requirements, `TestCases/` and optional `cache/` together in
`/content/drive/MyDrive/DataWork`. Setup mounts Drive when needed. If dependencies differ,
run `%pip install -r /content/drive/MyDrive/DataWork/requirements.txt`, restart the runtime,
then run all cells. Set `DATAWORKS_ROOT` or `PROJECT_ROOT_OVERRIDE` to choose another root.
No GPU is required. The first uncached grid extraction is slower than later cached runs.

**Sections:** 1. Setup; 2. Data loading; 3. Grid cleaning; 4. Training-only EDA;
5. Proxy evaluation; 6. History matching; 7. Twenty-year forecast and conditional uncertainty;
8. Deliverable; 9. Verification and business interpretation; 10. Submission checklist.

In [ ]:
# 1. Setup: one root for Windows, other local environments, and Google Colab
import importlib.metadata as metadata
import os
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None  # e.g. r'C:\path\to\dataworks-2026'; otherwise use DATAWORKS_ROOT or discover locally


def resolve_project_root(configured=None, *, cwd=None, in_colab=False, mount_drive=None):
    """An explicit root wins; otherwise mount Drive in Colab or find a local TestCases folder."""
    if configured:
        return Path(configured).expanduser().resolve()
    if in_colab:
        if mount_drive is None:
            raise ValueError('Colab discovery requires the Google Drive mount callback.')
        mount_drive('/content/drive')
        return Path('/content/drive/MyDrive/DataWork')
    current = Path.cwd() if cwd is None else Path(cwd)
    for candidate in (current, *current.parents):
        if (candidate / 'TestCases').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Set DATAWORKS_ROOT or PROJECT_ROOT_OVERRIDE to the folder containing TestCases/.')


try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive, IN_COLAB = None, False

PROJECT_ROOT = resolve_project_root(
    PROJECT_ROOT_OVERRIDE or os.environ.get('DATAWORKS_ROOT'),
    in_colab=IN_COLAB, mount_drive=drive.mount if IN_COLAB else None,
)
REQUIRED_PACKAGES = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'openpyxl',
                     'python-calamine', 'pyarrow', 'joblib', 'threadpoolctl']
try:
    PACKAGE_VERSIONS = {name: metadata.version(name) for name in REQUIRED_PACKAGES}
except metadata.PackageNotFoundError as exc:
    raise RuntimeError(f'Install {PROJECT_ROOT / "requirements.txt"} in this kernel, then restart it.') from exc
if PACKAGE_VERSIONS['scikit-learn'] != '1.6.1':
    raise RuntimeError('This project requires scikit-learn==1.6.1. Install requirements.txt and restart the kernel.')

import numpy as np
import pandas as pd
import scipy.linalg  # load SciPy's BLAS library before applying thread limits
import sklearn
from threadpoolctl import threadpool_limits

# These small Gaussian-process fits are faster and more reproducible with one BLAS thread.
_thread_limits = threadpool_limits(limits=1)
DATA_DIR = PROJECT_ROOT / 'TestCases'
CACHE_DIR = PROJECT_ROOT / 'cache'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
for directory in (CACHE_DIR, OUTPUT_DIR, FIGURE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

INTRO_FILE = DATA_DIR / '01 Introduction.xlsx'
FIELD_FILE = DATA_DIR / '02 Field Production.xlsx'
GRID_FILES = sorted(DATA_DIR.glob('0[3-8]*.xlsx'))
for required in (INTRO_FILE, FIELD_FILE, DATA_DIR / '09 Template Deliverable.xlsx'):
    if not required.is_file():
        raise FileNotFoundError(f'Missing required workbook: {required}')
SEED = 42
PARAMS = ['fault_trans', 'poro_mult', 'perm_mult', 'aquifer_pv']
CUMULATIVE_SHEETS = {'oil': 'Oil production cumulative', 'water': 'Water production cumulative',
                     'gas': 'Gas production cumulative'}
OBSERVED_COLUMN = 'RAMP_History'
print(f'Project root: {PROJECT_ROOT}; runtime: {"Colab" if IN_COLAB else "local"}')
print('Package versions:', PACKAGE_VERSIONS)
%matplotlib inline

## 2. Data loading

- **Parameters**: 4 uncertain inputs for each of the 100 simulation cases (1-70 train, 71-85 validation, 86-100 test).
- **Curves**: field oil, water and gas production for every case and for the observed history, 1998-2008.

Cleaning decisions:

1. The simulator reports each case on its own time steps (156-197 per case), so every cumulative curve is interpolated onto the 41 quarterly dates of the observed history.
2. Only the **cumulative** sheets are read. The simulator stamps a rate at the end of the period it covers while the observed table stamps it at the start, so the two rate tables are one quarter apart. Rates are instead derived from the cumulative curves, which puts simulated and observed data on exactly the same footing.
3. The result is cached as Parquet because the Excel workbook is slow to read.

In [ ]:
def load_parameters():
    """Return the 4 uncertain parameters per case, with the train/validation/test split."""
    df = pd.read_excel(INTRO_FILE, sheet_name='Uncertainty Values')
    df.columns = ['case'] + PARAMS
    df['case'] = df['case'].str.extract(r'(\d+)', expand=False).astype(int)
    df['split'] = pd.cut(df['case'], [0, 70, 85, 100], labels=['train', 'validation', 'test']).astype(str)
    return df.set_index('case')


def load_field_table():
    """Return the observed quarterly rate table, including water injection."""
    df = pd.read_excel(FIELD_FILE, sheet_name='Field Production').drop(columns='Identifier')
    df.columns = ['date', 'oil_rate', 'gas_rate', 'water_rate', 'water_injection_rate']
    return df.set_index('date')


def days_since(dates, start):
    """Convert dates to days since `start`, as floats for interpolation."""
    return np.asarray((pd.to_datetime(dates) - start) / pd.Timedelta(days=1), dtype=float)


def load_cumulatives(dates):
    """Return simulated (long table) and observed (wide table) cumulative production on `dates`."""
    sim_path, obs_path = CACHE_DIR / 'cumulatives.parquet', CACHE_DIR / 'observed_cumulatives.parquet'
    if sim_path.exists() and obs_path.exists():
        return pd.read_parquet(sim_path), pd.read_parquet(obs_path)

    start = dates[0]
    book = pd.read_excel(FIELD_FILE, sheet_name=list(CUMULATIVE_SHEETS.values()), header=1)

    simulated, observed = [], pd.DataFrame(index=dates)
    for fluid, sheet in CUMULATIVE_SHEETS.items():
        raw = book[sheet].iloc[1:]  # the first row holds unit labels
        t = days_since(raw['Date'], start)
        for column in raw.columns[1:]:
            y = pd.to_numeric(raw[column], errors='coerce').to_numpy()
            reported = ~np.isnan(y)
            values = np.interp(days_since(dates, start), t[reported], y[reported])
            if column == OBSERVED_COLUMN:
                observed[fluid] = values
            else:
                simulated.append(pd.DataFrame({
                    'case': int(column.split('_')[1]), 'fluid': fluid, 'date': dates, 'value': values,
                }))

    simulated = pd.concat(simulated, ignore_index=True)
    simulated.to_parquet(sim_path)
    observed.to_parquet(obs_path)
    return simulated, observed


def quarterly_rates(cumulative):
    """Average daily rate over the quarter ending at each date, from cumulative differences."""
    days = cumulative.index.to_series().diff().dt.days
    return cumulative.diff().div(days, axis=0).fillna(0)


params = load_parameters()
field_table = load_field_table()
dates = field_table.index.rename('date')

cumulative_long, obs_cum = load_cumulatives(dates)
cum = cumulative_long.pivot(index='date', columns=['fluid', 'case'], values='value')  # cum['oil'] is dates x cases
rate, obs_rate = quarterly_rates(cum), quarterly_rates(obs_cum)

print(params['split'].value_counts().to_dict())
print(f'cumulative curves: {cum.shape[0]} dates x {cum["oil"].shape[1]} cases x {cum.columns.get_level_values("fluid").nunique()} fluids')
print(f'history: {dates[0].date()} to {dates[-1].date()}')

In [ ]:
# Data quality checks on the loaded tables
checks = {
    'cases with parameters': len(params),
    'missing parameter values': int(params[PARAMS].isna().sum().sum()),
    'duplicate parameter sets': int(params[PARAMS].duplicated().sum()),
    'missing curve values': int(cum.isna().sum().sum() + obs_cum.isna().sum().sum()),
    'negative curve values': int((cum < 0).sum().sum()),
    'cumulative curves that decrease': int((cum.diff() < 0).any().sum()),
}
for name, value in checks.items():
    print(f'{name}: {value}')

# Check the approximate gas/oil proportionality on training responses only.
training_ids = params.index[params['split'] == 'train']
gas_oil_ratio = cum['gas'][training_ids].iloc[-1] / cum['oil'][training_ids].iloc[-1]
print(f'gas/oil ratio across training cases: {gas_oil_ratio.min():.4f} to {gas_oil_ratio.max():.4f}')

# The observed rate table and the observed cumulative curve agree once each rate is applied to the quarter after its date
shifted_table_rate = field_table['oil_rate'].shift(1).fillna(0)
print(f'largest gap between the observed rate table and rates derived from cumulatives: '
      f'{(shifted_table_rate - obs_rate["oil"]).abs().max():.2f} STB/d')

## 3. Grid data cleaning

Each case also comes with a 53 x 55 x 31 grid snapshot (90,365 cells, 29 properties). Cleaning steps:

1. Shorten the column names to the simulator keywords (`PORV`, `PORO`, `PERMX`, ...).
2. Replace the placeholder `-999.25`, which marks inactive cells, with missing values.
3. Reduce every case to a few field-level numbers (pore volumes, average rock quality, initial state), because the proxy works on field totals and 100 cases are far too few to learn from 90,365 cells each.

The workbooks total 1.3 GB, so the summary is cached and the full grid is kept for case 1 only (for maps).

In [ ]:
from python_calamine import CalamineWorkbook

INACTIVE = -999.25


def short_name(header):
    """'Porosity (PORO)' -> 'PORO', 'I Index' -> 'I'."""
    header = str(header).strip()
    return header[header.rfind('(') + 1:-1] if header.endswith(')') else header.split()[0]


def read_grid(workbook, sheet):
    """Read one case sheet as a numeric table with inactive cells set to NaN."""
    rows = workbook.get_sheet_by_name(sheet).to_python()
    grid = pd.DataFrame(rows[1:], columns=[short_name(h) for h in rows[0]])
    grid = grid.apply(pd.to_numeric, errors='coerce')
    return grid.replace(INACTIVE, np.nan)


def summarise_grid(grid):
    """Field-level numbers describing one case's rock and initial state."""
    active = grid[grid['PORO'].notna()]
    oil_zone, water_zone = active[active['SOIL'] > 0], active[active['SOIL'] == 0]
    return {
        'active_cells': len(active),
        'pore_volume': active['PORV'].sum(),
        'oil_pore_volume': (active['PORV'] * active['SOIL']).sum(),
        'oil_zone_pore_volume': oil_zone['PORV'].sum(),
        'water_zone_pore_volume': water_zone['PORV'].sum(),
        'porosity_oil_zone': oil_zone['PORO'].mean(),
        'porosity_water_zone': water_zone['PORO'].mean(),
        'perm_oil_zone': np.exp(np.log(oil_zone['PERMX']).mean()),
        'perm_water_zone': np.exp(np.log(water_zone['PERMX']).mean()),
        'tranx_sum': active['TRANX'].sum(),
        'trany_sum': active['TRANY'].sum(),
        'tranz_sum': active['TRANZ'].sum(),
        'initial_pressure': active['PRESSURE'].mean(),
        'initial_water_saturation': active['SWAT'].mean(),
        'initial_gas_saturation': active['SGAS'].mean(),
    }


def load_grid_features():
    """Return one row of grid summary numbers per case, reading the workbooks only once."""
    path = CACHE_DIR / 'grid_features.parquet'
    if path.exists():
        return pd.read_parquet(path)

    records = []
    for file in GRID_FILES:
        workbook = CalamineWorkbook.from_path(file)
        for sheet in workbook.sheet_names:
            case = int(sheet.split()[1])
            grid = read_grid(workbook, sheet)
            if case == 1:
                grid.to_parquet(CACHE_DIR / 'grid_case_1.parquet')
            records.append({'case': case, 'rows': len(grid), **summarise_grid(grid)})
        print(f'{file.name}: {len(workbook.sheet_names)} cases read')

    features = pd.DataFrame(records).set_index('case').sort_index()
    features.to_parquet(path)
    return features


grid_features = load_grid_features()
print(grid_features.shape)

In [ ]:
# Response exploration is confined to the designated training cases.
def training_eda_tables(parameters, cumulative, rates, grids):
    training_ids = parameters.index[parameters['split'] == 'train']
    columns = cumulative.columns.get_level_values('case').isin(training_ids)
    return (parameters.loc[training_ids].copy(), cumulative.loc[:, columns].copy(),
            rates.loc[:, columns].copy(), grids.loc[training_ids].copy())


eda_params, eda_cum, eda_rate, eda_grid_features = training_eda_tables(params, cum, rate, grid_features)

# Checks on the grid summary: what is identical in every case, and what does each parameter change?
constant = [c for c in eda_grid_features if eda_grid_features[c].nunique() == 1]
varying = [c for c in eda_grid_features if c not in constant]
print('identical in the 70 training cases:', constant)

parameter_effect = (pd.concat([eda_params[PARAMS], eda_grid_features[varying]], axis=1)
                    .corr(method='spearman').loc[varying, PARAMS])
print('\nrank correlation between grid summary numbers and the 4 parameters:')
print(parameter_effect.round(2).to_string())

# Oil pore volume is the porosity multiplier times a constant, providing a static normalization coordinate (not stock-tank oil in place)
oil_pv_per_unit = eda_grid_features['oil_pore_volume'] / eda_params['poro_mult']
print(f'\noil pore volume / porosity multiplier: {oil_pv_per_unit.min():.4e} to {oil_pv_per_unit.max():.4e}')

## 4. Exploratory data analysis

Production-response charts and relationships below use only the 70 training cases. The schema overview and input-only split coverage may describe all 100 provided inputs. Earlier development versions inspected all cases; see the evaluation provenance note in section 5.

Each step answers one question. A chart's title states the answer, and the note under it says how to read it and what it means. The section ends with a table of the findings.

**Colours used in every figure:** green = oil, blue = water, red = gas, black = the real field, grey = simulated cases, orange = the thing the figure is pointing at.

| Part | Step | Question |
|---|---|---|
| A. The dataset | 4.1 | What is in the dataset? |
| | 4.2 | What does one case look like? |
| B. Data quality | 4.3 | Is the data clean, and does every column carry information? |
| C. One variable at a time | 4.4 | How is each parameter distributed? |
| | 4.5 | How are the outcomes distributed, and where is the real field? |
| | 4.6 | Do the three groups of cases cover the same parameter ranges? |
| D. Relationships | 4.7 | Are the parameters independent of each other? |
| | 4.8 | How does each parameter relate to the oil and water produced? |
| | 4.9 | How strong is each link, and what explains what porosity leaves? |
| | 4.10 | What do the observed totals say about porosity? |
| E. Behaviour over time | 4.11 | Does the real field lie inside the range of the simulated cases? |
| | 4.12 | What is the life story of the field? |
| | 4.13 | When does water arrive, and how much of the liquid is water? |
| | 4.14 | How many numbers does it take to describe a production curve? |
| F. Domain check | 4.15 | Does the data agree with reservoir physics? |

In [ ]:
import itertools
import textwrap

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# FIGURE_DIR is configured once in setup.

# One colour per meaning, the same in every figure
COLOURS = {'oil': '#1baf7a', 'water': '#2a78d6', 'gas': '#e34948',  # the three fluids
           'observed': '#0b0b0b',                                   # the real field
           'simulated': '#c3c2b7',                                  # simulated cases as background
           'highlight': '#eb6834',                                  # whatever the figure is about
           'neutral': '#52514e', 'muted': '#898781'}
SPLITS = ('train', 'validation', 'test')
PARAM_LABELS = {'fault_trans': 'Fault transmissibility', 'poro_mult': 'Porosity multiplier',
                'perm_mult': 'Permeability multiplier', 'aquifer_pv': 'Aquifer pore volume'}

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.edgecolor': '#c3c2b7', 'axes.titlesize': 10, 'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
})


def finish(fig, finding, name, legend_columns=4):
    """Write the finding above the figure, put the legend in a row under it, then save and show the figure.

    The legend sits outside the plot area so it can never cover data.
    """
    width, height = fig.get_size_inches()
    finding = textwrap.fill(finding, int(width * 10.5))
    title_height = 0.08 + 0.21 * (finding.count('\n') + 1)  # inches
    fig.suptitle(finding, x=0.01, y=1 - 0.06 / height, ha='left', va='top', fontsize=12, fontweight='bold')
    entries = {label: handle for ax in fig.axes for handle, label in zip(*ax.get_legend_handles_labels())}
    legend_height = 0
    if entries:
        fig.legend(entries.values(), entries.keys(), loc='upper left', bbox_to_anchor=(0.005, 1 - title_height / height),
                   ncol=legend_columns, frameon=False)
        legend_height = 0.27 * int(np.ceil(len(entries) / legend_columns))
    fig.tight_layout(pad=0.6, rect=(0, 0, 1, 1 - (title_height + legend_height) / height))
    fig.savefig(FIGURE_DIR / f'{name}.png')
    plt.show()


def draw_range(ax, curves, name='simulated cases'):
    """Grey bands for a set of curves (dates x cases): the full range and the middle 80%."""
    ax.fill_between(curves.index, curves.min(axis=1), curves.max(axis=1), color=COLOURS['simulated'], alpha=0.35, linewidth=0,
                    label=f'All {curves.shape[1]} {name}')
    ax.fill_between(curves.index, curves.quantile(0.1, axis=1), curves.quantile(0.9, axis=1), color=COLOURS['simulated'],
                    linewidth=0, label='Middle 80% of them')


# 4.1 What is in the dataset?
overview = pd.DataFrame([
    {'table': 'Parameters (model inputs)', 'rows': f'{len(params)} cases', 'columns': f'{len(PARAMS)} parameters + split label',
     'values': 'decimal numbers'},
    {'table': 'Simulated production (model outputs)', 'rows': f'{len(dates)} quarterly dates',
     'columns': f'{cum.shape[1]} = {len(params)} cases x {len(CUMULATIVE_SHEETS)} fluids', 'values': 'cumulative volumes, decimal numbers'},
    {'table': 'Observed production (real field)', 'rows': f'{len(obs_cum)} quarterly dates',
     'columns': f'{obs_cum.shape[1]} fluids', 'values': 'cumulative volumes, decimal numbers'},
    {'table': 'Grid summary (starting state of the rock)', 'rows': f'{len(grid_features)} cases',
     'columns': f'{grid_features.shape[1]} field totals', 'values': 'decimal numbers'},
]).set_index('table')
print(overview.to_string())
print(f'\nperiod covered: {dates[0]:%B %Y} to {dates[-1]:%B %Y}')
print('cases per split:', params['split'].value_counts()[list(SPLITS)].to_dict())
print('there are no text or category columns to model: every input and output is a number')

**How to read it.** Each row of the table is one of the four data tables this project uses, with its size.

**What it means.** This is a small, fully numeric dataset: 100 simulated cases, each with 4 input numbers and 3 production curves of 41 points, plus one real production history to compare against. There are no text or category columns, so no encoding is needed.

In [ ]:
# 4.2 What does one case look like?
example_case = (eda_params['poro_mult'] - eda_params['poro_mult'].median()).abs().idxmin()
input_lines = '\n'.join(f'{PARAM_LABELS[param]}: {eda_params.loc[example_case, param]:.3g}' for param in PARAMS)

fig, (ax, card) = plt.subplots(1, 2, figsize=(10, 4.2), width_ratios=[2.4, 1])
for fluid in ('oil', 'water'):
    ax.plot(eda_cum[fluid][example_case] / 1e6, color=COLOURS[fluid], linewidth=2.5, label=f'Cumulative {fluid}')
ax.set_ylabel('Million STB produced since the start')
card.axis('off')
card.text(0, 0.98, 'INPUTS: 4 numbers', fontweight='bold', va='top')
card.text(0, 0.88, input_lines, va='top', linespacing=1.7)
card.text(0, 0.40, 'OUTPUTS: 3 curves', fontweight='bold', va='top')
card.text(0, 0.30, f'Oil, water and gas, each with\n{len(dates)} quarterly values.\nGas is not drawn because it is\nmeasured in a different unit.', va='top', linespacing=1.7)
finish(fig, f'One case is 4 input numbers and 3 production curves (case {example_case} shown)', 'eda_02_one_case')

**How to read it.** The chart shows a single case: how much oil (green) and water (blue) it has produced in total by each date. The box on the right lists the 4 numbers that were fed into the simulator to produce these curves.

**What it means.** This is one "row" of the dataset. The proxy model has to learn the step from the 4 numbers on the right to the curves on the left.

In [ ]:
# 4.3 Is the data clean, and does every column carry information?
quality = pd.Series({
    'missing values in the parameters': eda_params[PARAMS].isna().sum().sum(),
    'missing values in the production curves': eda_cum.isna().sum().sum() + obs_cum.isna().sum().sum(),
    'duplicated parameter sets': eda_params[PARAMS].duplicated().sum(),
    'negative production values': (eda_cum < 0).sum().sum() + (obs_cum < 0).sum().sum(),
    'cumulative curves that ever decrease': (eda_cum.diff().iloc[1:] < 0).any().sum(),
}, name='count')
print(quality.to_frame().assign(result=np.where(quality == 0, 'pass', 'CHECK')).to_string())

gas_oil_ratio = (eda_cum['gas'] / eda_cum['oil']).where(eda_cum['oil'] > 0).stack()
print(f'\ngas divided by oil, all cases and dates: {gas_oil_ratio.min():.4f} to {gas_oil_ratio.max():.4f} MSCF per STB')

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(eda_cum['oil'] / 1e6, eda_cum['gas'] / 1e6, s=10, color=COLOURS['gas'], alpha=0.5, linewidth=0, rasterized=True,
           label=f'Every date of every simulated case ({eda_cum["oil"].size:,} points)')
ax.plot(obs_cum['oil'] / 1e6, obs_cum['gas'] / 1e6, color=COLOURS['observed'], linewidth=1.5, label='Real field')
ax.set_xlabel('Cumulative oil [million STB]')
ax.set_ylabel('Cumulative gas [million MSCF]')
finish(fig, f'Training gas is approximately {gas_oil_ratio.median():.4f} x oil, oil; gas is derived for this dataset', 'eda_03_gas_follows_oil')

**How to read it.** The quality table and simulated dots use training production responses.
The black line shows the provided observed field. Reported counts and gas/oil ratios are
calculated at execution time.

**Interpretation.** Gas is approximately proportional to oil in this training dataset.
Gas is therefore derived for this workflow, with units and the observed field ratio stated
explicitly. Approximate proportionality is not a universal reservoir-physics rule.

In [ ]:
# 4.4 How is each parameter distributed?
statistics = eda_params[PARAMS].describe().T[['mean', '50%', 'std', 'min', '25%', '75%', 'max']].rename(columns={'50%': 'median'})
statistics['IQR'] = statistics['75%'] - statistics['25%']
statistics['skew'] = eda_params[PARAMS].skew()
print(statistics.round(3).to_string())

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
for ax, param in zip(axes, PARAMS):
    ax.hist(eda_params[param], bins=10, color=COLOURS['simulated'], edgecolor='white')
    ax.axvline(eda_params[param].median(), color=COLOURS['highlight'], linewidth=2, label='Median (half of the cases on each side)')
    ax.set_title(PARAM_LABELS[param])
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('Number of cases')
finish(fig, 'Each parameter is spread evenly over its range: no clusters, no gaps, no skew', 'eda_04_parameter_distributions')

**How to read it.** Each histogram describes one input across the 70 training cases; the
orange line marks its median. The table reports spread and skew.

**Interpretation.** These are descriptive summaries. A visually broad marginal range does
not prove joint coverage of parameter space; transformations and model performance are
assessed through the training/validation protocol in section 5.

In [ ]:
# 4.5 How are the outcomes distributed, and where is the real field?
WATER_BREAKTHROUGH_RATE = 100  # STB/d of field water that counts as "water has arrived"
first_oil_date = eda_rate.index[(eda_rate['oil'] > 0).any(axis=1)][0]


def breakthrough_years(water_rate):
    """Years from first production until the water rate first exceeds the threshold."""
    arrived = water_rate[water_rate > WATER_BREAKTHROUGH_RATE]
    return (arrived.index[0] - first_oil_date).days / 365.25 if len(arrived) else np.nan


liquid_rate = eda_rate['oil'] + eda_rate['water']
water_cut = (eda_rate['water'] / liquid_rate).where(liquid_rate > 0)
obs_water_cut = obs_rate['water'] / (obs_rate['oil'] + obs_rate['water'])

case_summary = pd.DataFrame({
    'oil_cum_2008': eda_cum['oil'].iloc[-1],
    'water_cum_2008': eda_cum['water'].iloc[-1],
    'oil_rate_2008': eda_rate['oil'].iloc[-1],
    'water_cut_2008': water_cut.iloc[-1],
    'water_breakthrough_years': eda_rate['water'].apply(breakthrough_years),
})

outcomes = {  # label: (values of the 70 training cases, value of the real field, colour)
    'Cumulative oil in 2008 [million STB]': (case_summary['oil_cum_2008'] / 1e6, obs_cum['oil'].iloc[-1] / 1e6, COLOURS['oil']),
    'Cumulative water in 2008 [million STB]': (case_summary['water_cum_2008'] / 1e6, obs_cum['water'].iloc[-1] / 1e6, COLOURS['water']),
    'Water cut in 2008 [% of produced liquid]': (100 * case_summary['water_cut_2008'], 100 * obs_water_cut.iloc[-1], COLOURS['water']),
}

n_outliers = 0
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, (label, (values, field_value, colour)) in zip(axes, outcomes.items()):
    q1, q3 = values.quantile([0.25, 0.75])
    n_outliers += ((values < q1 - 1.5 * (q3 - q1)) | (values > q3 + 1.5 * (q3 - q1))).sum()
    ax.boxplot(values, vert=False, widths=0.5, patch_artist=True, boxprops={'facecolor': 'white', 'edgecolor': COLOURS['neutral']},
               medianprops={'color': COLOURS['neutral'], 'linewidth': 2}, whiskerprops={'color': COLOURS['neutral']},
               capprops={'color': COLOURS['neutral']}, flierprops={'markeredgecolor': COLOURS['highlight']})
    ax.scatter(values, 1 + rng.uniform(-0.18, 0.18, len(values)), s=14, color=colour, alpha=0.6, linewidth=0, zorder=3)
    ax.axvline(field_value, color=COLOURS['observed'], linewidth=2.5, label='Real field')
    ax.set_title(label)
    ax.set_yticks([])
    ax.grid(axis='y', visible=False)
axes[0].scatter([], [], s=14, color=COLOURS['neutral'], label='One simulated case')
axes[0].plot([], [], color=COLOURS['neutral'], linewidth=1, label='Box: middle half of the cases, with the median inside')
finish(fig, f'{n_outliers or "No"} outliers among the 70 training cases; the real field sits on the high-oil, low-water side', 'eda_05_outcome_distributions')
print('whiskers reach the furthest case within 1.5 box lengths of the box; a case beyond them would count as an outlier')

**How to read it.** Each dot is one simulated case. The box covers the middle half of the cases, with the median as the line inside it. The thick black line is the real field.

**What it means.** No case lies far from the rest, so none has to be removed. The real field sits to the right for oil and to the left for water: it behaves like the cases that produce more oil and less water than average.

In [ ]:
# 4.6 Do the three groups of cases cover the same parameter ranges?
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, param in zip(axes, PARAMS):
    for row, split in enumerate(SPLITS):
        values = params.loc[params['split'] == split, param]
        low, q1, median, q3, high = values.quantile([0, 0.25, 0.5, 0.75, 1])
        ax.barh(row, high - low, left=low, height=0.34, color=COLOURS['highlight'], alpha=0.3, label='Full range: lowest to highest case')
        ax.barh(row, q3 - q1, left=q1, height=0.34, color=COLOURS['highlight'], label='Middle half of the cases')
        ax.plot(median, row, marker='|', markersize=18, markeredgewidth=2.5, linestyle='', color=COLOURS['neutral'], label='Median')
    ax.set_title(PARAM_LABELS[param])
    ax.grid(axis='y', visible=False)
axes[0].set_yticks(range(len(SPLITS)), [f'{split} ({(params["split"] == split).sum()} cases)' for split in SPLITS])
axes[0].invert_yaxis()
finish(fig, 'Provided input ranges by split; overlapping marginal ranges do not prove joint coverage', 'eda_06_split_coverage')

**How to read it.** Each panel is one parameter, and each of its three bars is one group of cases. The pale bar runs from the group's lowest to its highest value, the solid part holds the middle half of its cases, and the dark tick is its median.

**What it means.** In every panel the three bars start and end at almost the same place and their medians sit close together. The marginal ranges overlap, although that alone does not prove joint coverage of the four-dimensional parameter space. Held-out prediction errors are checked in section 5.

In [ ]:
# 4.7 Are the parameters independent of each other?
input_correlation = eda_params[PARAMS].corr(method='spearman')
pairs = pd.Series({f'{PARAM_LABELS[a]}\nand {PARAM_LABELS[b].lower()}': input_correlation.loc[a, b]
                   for a, b in itertools.combinations(PARAMS, 2)}).round(2).sort_values(key=abs)

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.barh(pairs.index, pairs, height=0.6, color=COLOURS['neutral'])
ax.bar_label(bars, labels=[f'{abs(value):.2f}' if value == 0 else f'{value:+.2f}' for value in pairs], padding=5)
ax.axvline(0, color=COLOURS['muted'], linewidth=1)
ax.set_xlim(-1, 1)
ax.set_xticks([-1, -0.5, 0, 0.5, 1], ['-1\nalways opposite', '-0.5', '0\nno link', '0.5', '+1\nalways together'])
ax.grid(axis='y', visible=False)
finish(fig, f'Training inputs: strongest pairwise rank association is {pairs.abs().max():.2f}', 'eda_07_parameter_independence')

**How to read it.** Each bar is a pairwise Spearman correlation within the training inputs.

**Interpretation.** Weak pairwise rank correlations do not establish statistical independence,
exclude interactions, or make an outcome association causal. The calculated values support
descriptive comparisons only.

In [ ]:
# 4.8 How does each parameter relate to the oil and water produced by 2008?
fig, axes = plt.subplots(2, 4, figsize=(14, 5.6), sharex='col', sharey='row')
for row, fluid in zip(axes, ('oil', 'water')):
    for ax, param in zip(row, PARAMS):
        ax.scatter(eda_params[param], case_summary[f'{fluid}_cum_2008'] / 1e6, s=16, color=COLOURS[fluid], alpha=0.7, linewidth=0,
                   label=f'One simulated case, {fluid}')
        ax.axhline(obs_cum[fluid].iloc[-1] / 1e6, color=COLOURS['observed'], linewidth=1.5, label='Real field')
    row[0].set_ylabel(f'Cumulative {fluid} in 2008\n[million STB]')
for ax, param in zip(axes[1], PARAMS):
    ax.set_xlabel(PARAM_LABELS[param])
finish(fig, 'Only porosity lines the cases up; against the other three parameters they form shapeless clouds', 'eda_08_parameters_vs_outcomes')

**How to read it.** Each small chart puts one parameter on the horizontal axis and the 2008 total on the vertical axis (oil on top, water below). One dot is one case, and the black line is the real field. Dots in a tight line mean the parameter controls the outcome; a shapeless cloud means it does not.

**What it means.** Only the porosity column forms a line. The link is slightly curved, not straight, so the proxy must be able to follow a curve.

In [ ]:
# 4.9 How strong is each link, and what explains the differences that porosity leaves?
def grouped_bars(ax, table):
    """Horizontal bars for a parameters x fluids table of link strengths, one colour per fluid."""
    positions, bar_height = np.arange(len(table)), 0.36
    for offset, fluid in zip((bar_height / 2, -bar_height / 2), table.columns):
        bars = ax.barh(positions + offset, table[fluid], height=bar_height - 0.04, color=COLOURS[fluid], label=f'Cumulative {fluid} in 2008')
        ax.bar_label(bars, fmt='{:.2f}', padding=4)
    ax.set_yticks(positions, [PARAM_LABELS[param] for param in table.index])
    ax.set_xlim(0, 1.1)
    ax.set_xticks([0, 0.5, 1], ['0\nno link', '0.5', '1\nstrong rank association'])
    ax.grid(axis='y', visible=False)


driver_table = (pd.concat([eda_params[PARAMS], case_summary], axis=1)
                .corr(method='spearman').loc[PARAMS, case_summary.columns])
print('Spearman rank correlation between each parameter and each outcome (the sign gives the direction):')
print(driver_table.round(2).to_string())

strength = driver_table[['oil_cum_2008', 'water_cum_2008']].abs().set_axis(['oil', 'water'], axis=1).sort_values('oil')
fig, ax = plt.subplots(figsize=(9, 3.6))
grouped_bars(ax, strength)
finish(fig, 'Porosity has the strongest training association with oil and water totals', 'eda_09_link_strength')

# Remove the porosity effect with a smooth curve, then ask which parameter explains what is left
other_params = [param for param in PARAMS if param != 'poro_mult']
leftover, share_left = {}, {}
for fluid in ('oil', 'water'):
    outcome = case_summary[f'{fluid}_cum_2008']
    porosity_fit = np.polyval(np.polyfit(eda_params['poro_mult'], outcome, 2), eda_params['poro_mult'])
    share_left[fluid] = (outcome - porosity_fit).var() / outcome.var()
    leftover[fluid] = eda_params[other_params].corrwith(outcome - porosity_fit, method='spearman').abs()
leftover = pd.DataFrame(leftover).sort_values('oil')
print(f'\nshare of the case-to-case differences that porosity does not explain: '
      f'oil {share_left["oil"]:.1%}, water {share_left["water"]:.1%}')

fig, ax = plt.subplots(figsize=(9, 3.2))
grouped_bars(ax, leftover)
finish(fig, f'In the small part porosity leaves unexplained, {PARAM_LABELS[leftover["oil"].idxmax()].lower()} is associated with oil '
            f'and {PARAM_LABELS[leftover["water"].idxmax()].lower()} drives water', 'eda_10_second_order_drivers')

**How to read it.** The first chart compares rank associations in the training cases. The
second compares associations after subtracting a quadratic porosity fit. The residual
variance shares and correlations are printed by the preceding cell.

**Interpretation.** Strong porosity association motivates a compact nonlinear proxy. Residual
associations suggest additional influences; they do not prove causal effects or determine
which parameters are uniquely identifiable from the observed history.

In [ ]:
# 4.10 What do the observed totals say about porosity? Read each fitted line at the observed value
implied_porosity = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, fluid in zip(axes, ('oil', 'water')):
    total, field_total = case_summary[f'{fluid}_cum_2008'] / 1e6, obs_cum[fluid].iloc[-1] / 1e6
    slope, intercept = np.polyfit(eda_params['poro_mult'], total, 1)
    implied_porosity[fluid] = (field_total - intercept) / slope
    ax.scatter(eda_params['poro_mult'], total, s=22, color=COLOURS[fluid], alpha=0.7, linewidth=0)
    ax.axhline(field_total, color=COLOURS['observed'], linewidth=2, label='Real field')
    ax.axvline(implied_porosity[fluid], color=COLOURS['highlight'], linewidth=2, linestyle='--',
               label='Porosity multiplier at which the cases reach the real field')
    ax.annotate(f'{implied_porosity[fluid]:.2f}', (implied_porosity[fluid], ax.get_ylim()[0]), xytext=(6, 6), textcoords='offset points',
                color=COLOURS['neutral'], fontweight='bold')
    ax.set_xlabel('Porosity multiplier')
    ax.set_title(f'Cumulative {fluid} in 2008 [million STB], one dot per simulated case')
finish(fig, f'Oil and water both point to a porosity multiplier of about {np.mean(list(implied_porosity.values())):.2f}', 'eda_11_porosity_from_observed')

**How to read it.** Training-case totals are regressed on porosity and compared with the
observed field totals.

**Interpretation.** The intersections provide approximate one-variable diagnostics. They
are not the solution of the joint oil/water and rate/cumulative history-match objective;
section 6 performs that optimization and reports its remaining residuals.

In [ ]:
# 4.11 Can the real field be matched at all? Only if it lies inside the range of the simulated cases
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, fluid in zip(axes, ('oil', 'water')):
    simulated, field_curve = eda_cum[fluid] / 1e6, obs_cum[fluid] / 1e6
    draw_range(ax, simulated)
    ax.plot(field_curve, color=COLOURS['observed'], linewidth=2, label='Real field')
    share_below = (simulated.iloc[-1] < field_curve.iloc[-1]).mean()
    ax.text(0.98, 0.05, f'Real field in 2008: {field_curve.iloc[-1]:.1f} million STB,\nmore than {share_below:.0%} of the cases',
            transform=ax.transAxes, ha='right', va='bottom')
    ax.set_title(f'Cumulative {fluid} [million STB]')
finish(fig, 'The real field lies inside the simulated range, with more oil and less water than most cases', 'eda_12_field_inside_range')

tolerance = 1e-6  # relative, so rounding does not count as "outside"
inside = {fluid: ((obs_cum[fluid] >= eda_cum[fluid].min(axis=1) * (1 - tolerance))
                  & (obs_cum[fluid] <= eda_cum[fluid].max(axis=1) * (1 + tolerance))).mean() for fluid in CUMULATIVE_SHEETS}
print('share of dates on which the real field is inside the simulated range:')
print(pd.Series(inside).round(2).to_string())

**How to read it.** The bands describe the 70 training histories: their full range and
middle 80%. The black line is the observed field. The preceding cell reports envelope
coverage for each fluid.

**Interpretation.** Being inside a pointwise envelope supports attempting history matching,
but does not prove that one static parameter set matches all dates and fluids together.

In [ ]:
# 4.12 What is the life story of the field? Oil rate over time
PLATEAU_SHARE = 0.95  # the plateau lasts while the rate stays within 5% of its peak
field_oil_rate = obs_rate['oil']
plateau_rate = round(field_oil_rate.max(), -2)
plateau_end = field_oil_rate[field_oil_rate >= PLATEAU_SHARE * field_oil_rate.max()].index[-1]
final_rate = round(field_oil_rate.iloc[-1], -2)

fig, ax = plt.subplots(figsize=(10, 4.4))
draw_range(ax, eda_rate['oil'])
ax.plot(field_oil_rate, color=COLOURS['observed'], linewidth=2, label='Real field')
ax.axvline(plateau_end, color=COLOURS['highlight'], linewidth=1.5, linestyle='--', label='End of the plateau')
ax.set_ylim(0, plateau_rate * 1.22)
label_height = plateau_rate * 1.12
ax.text(first_oil_date + (plateau_end - first_oil_date) / 2, label_height, f'PLATEAU\nabout {plateau_rate:,.0f} STB/d', ha='center', va='center')
ax.text(plateau_end + (dates[-1] - plateau_end) / 2, label_height, f'DECLINE\ndown to about {final_rate:,.0f} STB/d', ha='center', va='center')
ax.set_ylabel('Oil rate [STB per day]')
finish(fig, f'The real field held about {plateau_rate:,.0f} STB/d until {plateau_end.year}, then fell to '
            f'{final_rate / plateau_rate:.0%} of that by 2008', 'eda_13_oil_rate_phases')

**How to read it.** The black line is the real field's oil rate. The grey bands show what the simulated cases did at the same time. The dashed line is where the real field's rate dropped more than 5% below its peak.

**What it means.** The field has two phases. Until 2002 it held a steady plateau of about 15,500 STB/d, and every case does the same, so this period tells us nothing about the parameters. After 2002 the rate falls and the cases fan out: the decline is where the information for history matching sits, and the starting point for the forecast.

In [ ]:
# 4.13 When does water arrive, and how much of the produced liquid is water?
breakthrough = case_summary['water_breakthrough_years']
final_cut = 100 * case_summary['water_cut_2008']

fig, ax = plt.subplots(figsize=(10, 4.4))
draw_range(ax, 100 * water_cut)
ax.plot(100 * obs_water_cut, color=COLOURS['observed'], linewidth=2, label='Real field')
ax.axhline(50, color=COLOURS['highlight'], linewidth=1.5, linestyle='--', label='Half of the liquid is water')
ax.set_ylabel('Water cut [% of produced liquid]')
ax.set_ylim(0, 100)
finish(fig, f'Water arrives within {breakthrough.max():.1f} years in every case; by 2008 it is {final_cut.min():.0f}-{final_cut.max():.0f}% '
            f'of the liquid ({100 * obs_water_cut.iloc[-1]:.0f}% in the real field)', 'eda_14_water_cut')

print(f'water breakthrough, years after first oil: cases {breakthrough.min():.2f} to {breakthrough.max():.2f}, '
      f'real field {breakthrough_years(obs_rate["water"]):.2f}')
print(f'water cut in the last quarter [%]: cases {final_cut.min():.1f} to {final_cut.max():.1f}, real field {100 * obs_water_cut.iloc[-1]:.1f}')

**How to read it.** Water cut is the share of water in the liquid that comes out of the wells. The grey bands are the simulated cases, the black line is the real field, and the dashed line marks the point where half of the liquid is water.

**What it means.** Water shows up 0.75 to 1.5 years after first oil in every case (1.25 years in the real field), so the arrival time does not separate the cases. What separates them is how fast the water cut rises. The real field crossed 50% in 2006 and is at 60% in 2008: most of what the wells lift is now water.

In [ ]:
# 4.14 How many numbers does it take to describe a curve? Rebuild every curve from a few PCA numbers and measure the error
MAX_NUMBERS = 5
train_cases = eda_params.index[eda_params['split'] == 'train']
rebuild_error = {}
for fluid in ('oil', 'water'):
    curves, train_curves = eda_cum[fluid].to_numpy().T, eda_cum[fluid][train_cases].to_numpy().T
    errors = []
    for n_numbers in range(1, MAX_NUMBERS + 1):
        pca = PCA(n_numbers).fit(train_curves)
        rebuilt = pca.inverse_transform(pca.transform(curves))
        errors.append(100 * np.sqrt(((rebuilt - curves) ** 2).mean()) / curves[:, -1].mean())
    rebuild_error[fluid] = errors
rebuild_error = pd.DataFrame(rebuild_error, index=range(1, MAX_NUMBERS + 1))
print('typical error of a curve rebuilt from n numbers [% of its 2008 value]:')
print(rebuild_error.round(3).to_string())

bar_width = 0.38
fig, ax = plt.subplots(figsize=(9, 3.8))
for offset, fluid in zip((-bar_width / 2, bar_width / 2), rebuild_error.columns):
    bars = ax.bar(rebuild_error.index + offset, rebuild_error[fluid], width=bar_width - 0.04, color=COLOURS[fluid], label=f'Cumulative {fluid} curve')
    ax.bar_label(bars, fmt='{:.2f}%', padding=3, fontsize=9)
ax.set_xticks(rebuild_error.index, [f'{n} number{"s" if n > 1 else ""}' for n in rebuild_error.index])
ax.set_xlabel('Numbers used to describe one 41-point curve')
ax.set_ylabel('Rebuild error [% of 2008 value]')
ax.set_ylim(0, rebuild_error.to_numpy().max() * 1.15)
ax.grid(axis='x', visible=False)
finish(fig, f'A 41-point curve can be rebuilt from {MAX_NUMBERS} numbers with an error of {rebuild_error.loc[MAX_NUMBERS].max():.2f}% or less', 'eda_15_curve_complexity')

**How to read it.** PCA is fitted to the training curves and the chart reports training
reconstruction error for each number of retained scores.

**Interpretation.** Low reconstruction error supports curve compression. It does not by
itself establish proxy generalization or that 70 cases are sufficient; section 5 separately
measures validation and reserved-case prediction errors.

In [ ]:
# 4.15 Does the data agree with reservoir physics?
sanity = pd.Series({
    'cumulative production never decreases': bool((eda_cum.diff().iloc[1:] >= 0).all().all()),
    'no production rate is negative': bool((eda_rate.fillna(0) >= 0).all().all()),
    'water arrives after oil starts, in every case': bool((breakthrough > 0).all()),
    'more pore space (porosity) gives more oil': driver_table.loc['poro_mult', 'oil_cum_2008'] > 0,
    'more pore space (porosity) gives a lower water cut': driver_table.loc['poro_mult', 'water_cut_2008'] < 0,
    'gas and oil keep a constant ratio (variation under 0.1%)': gas_oil_ratio.max() / gas_oil_ratio.min() - 1 < 1e-3,
    'the real field is inside the simulated range on every date': all(share == 1 for share in inside.values()),
}, name='result').map({True: 'pass', False: 'FAIL'})
print(sanity.to_string())

**How to read it.** Each line is a rule that reservoir physics says must hold. "pass" means the data obeys it.

**What it means.** These checks support physical plausibility of this synthetic waterflood dataset. They do not establish causal relationships, exact history-match feasibility, or twenty-year forecast accuracy.

### Implications for modelling

| Training analysis | Implication and limit |
|---|---|
| Data quality and input distributions | Inspect the calculated diagnostics; do not infer joint coverage from marginal ranges |
| Approximate gas/oil proportionality | Derive gas with stated units for this dataset; do not count a redundant target twice |
| Production/parameter associations | Motivate a nonlinear proxy; correlations alone do not prove causal effects or identifiability |
| Observed field versus training envelope | Attempt joint history matching, then report actual residuals; envelope coverage is not proof of a match |
| Recent oil decline and increasing water share | Evaluate forecast assumptions and water-handling quantities explicitly |
| Low curve reconstruction error | Use PCA compression and assess predictive accuracy on cases excluded from fitting |
| Physical sanity checks | Check plausibility without treating passing checks as proof of twenty-year accuracy |

## 5. Proxy model

The proxy replaces the reservoir simulator: it takes the 4 parameters and returns the cumulative oil and water curves in milliseconds.

- **Targets**: cumulative oil and cumulative water on the 41 quarterly dates. Gas is oil times a fixed ratio, so it is derived, not modelled.
- **Architecture**: each curve is compressed with PCA into a few scores, and one regressor per score maps the parameters to it (diagram in 5.8).
- **Candidates**: Gaussian process, quadratic response surface, reduced quadratic, neural network and random forest.
- **How "good" is measured**: errors on cases the model was not fitted on, as a percentage of the 2008 value. The table reports R2, RMSE (root mean squared error, punishes large misses), MAE (mean absolute error, the typical miss) and MAPE of the 2008 value. Models are chosen on RMSE.

**Protocol**

| Step | Cases used | Purpose |
|---|---|---|
| 5.1-5.2 | Fit on train (1-70), score on validation (71-85) | Choose the type of model |
| 5.3 | 5-fold cross-validation inside train (1-70) | Fine-tune the chosen model |
| 5.4 | Fit on train, score on validation (71-85) | Diagnose the tuned model without inspecting test responses |
| 5.5 | Fit on train, compare the error on train with the error on validation | Overfitting check |
| 5.6 | Fit on part of train, score on validation | Check that 70 training cases are enough |
| 5.7 | Freeze settings, fit on train + validation (1-85), score on test (86-100) | Final held-out proxy assessment and saved model |
| 5.8 | - | Diagram of the final proxy |

Test responses are excluded from fitting and tuning. Section 5.7 evaluates the frozen proxy; section 7 separately evaluates the frozen forecast method on held-out test cases. Parameter bounds use supplied input values, not test production targets.

**Evaluation provenance.** Production-response EDA in this version is restricted to
training. Earlier development versions inspected all 100 cases and test diagnostics.
Reserved-case scores are not a prospectively blind assessment. No test production targets
are fitted or used for the current hyperparameter selection. Code and documentation
development included OpenAI Codex assistance; actual contributions are not invented.

In [ ]:
import time
import warnings

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# A parameter with almost no effect pushes its length scale to the upper bound, and sklearn warns on every fit
warnings.filterwarnings('ignore', message='The optimal value found for dimension', category=ConvergenceWarning)

MODELLED_FLUIDS = ['oil', 'water']
CASES = {split: params.index[params['split'] == split] for split in SPLITS}
ERROR_METRIC = 'curve RMSE [% of 2008 value]'  # the metric used to choose between models


class CurveProxy:
    """Predict one fluid's cumulative curve from the 4 parameters.

    The curve is compressed with PCA into a few scores and one regressor is trained per score.
    Parameters listed in `log_columns` enter the model as their logarithm.
    """

    def __init__(self, regressor, variance=0.9999, log_columns=()):
        self.regressor = regressor
        self.variance = variance
        self.log_columns = list(log_columns)

    def _prepare(self, X):
        X = np.array(X, dtype=float)
        X[:, self.log_columns] = np.log(X[:, self.log_columns])
        return X

    def fit(self, X, Y):
        self.x_scaler = StandardScaler().fit(self._prepare(X))
        self.pca = PCA(n_components=self.variance, svd_solver='full').fit(Y)
        scores = self.pca.transform(Y)
        X_scaled = self.x_scaler.transform(self._prepare(X))
        self.models = [clone(self.regressor).fit(X_scaled, scores[:, k]) for k in range(scores.shape[1])]
        return self

    def predict(self, X, return_std=False):
        X_scaled = self.x_scaler.transform(self._prepare(X))
        if not return_std:
            scores = np.column_stack([model.predict(X_scaled) for model in self.models])
            return self._physical(self.pca.inverse_transform(scores))
        results = [model.predict(X_scaled, return_std=True) for model in self.models]
        scores = np.column_stack([mean for mean, _ in results])
        score_std = np.column_stack([std for _, std in results])
        curve_std = np.sqrt(score_std ** 2 @ self.pca.components_ ** 2)
        return self._physical(self.pca.inverse_transform(scores)), curve_std

    @staticmethod
    def _physical(curves):
        """Cumulative production can be neither negative nor decreasing."""
        return np.maximum.accumulate(np.clip(curves, 0, None), axis=1)


def inputs(cases):
    return params.loc[cases, PARAMS].to_numpy()


def targets(cases, fluid):
    """Cumulative curves as a cases x dates array."""
    return cum[fluid][cases].to_numpy().T


def score(proxy, cases, fluid):
    """Accuracy of a proxy on cases it was not fitted on. Errors are in percent so oil and water can be compared."""
    actual, predicted = targets(cases, fluid), proxy.predict(inputs(cases))
    error = predicted - actual
    final_value = actual[:, -1].mean()
    return {
        'R2': 1 - (error ** 2).sum() / ((actual - actual.mean(axis=0)) ** 2).sum(),
        ERROR_METRIC: 100 * np.sqrt((error ** 2).mean()) / final_value,          # root mean squared error, penalises large misses
        'curve MAE [% of 2008 value]': 100 * np.abs(error).mean() / final_value,  # mean absolute error, the typical miss
        '2008 value MAPE [%]': 100 * np.abs(error[:, -1] / actual[:, -1]).mean(),  # mean absolute percentage error of the end point
    }


MATERN_SMOOTHNESS = {'Matern 5/2': 2.5, 'Matern 3/2': 1.5}  # the higher the number, the smoother the assumed response


def gaussian_process(kernel='RBF'):
    """Gaussian process with one length scale per parameter and a noise term.

    'RBF' assumes a very smooth response; 'Matern 5/2' allows a slightly rougher one and 'Matern 3/2' a rougher one still.
    """
    lengths = {'length_scale': np.ones(len(PARAMS)), 'length_scale_bounds': (1e-2, 1e3)}
    shape = RBF(**lengths) if kernel == 'RBF' else Matern(nu=MATERN_SMOOTHNESS[kernel], **lengths)
    noise = WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1))
    return GaussianProcessRegressor(ConstantKernel() * shape + noise, normalize_y=True, n_restarts_optimizer=3, random_state=SEED)


def quadratic(final_step):
    """Response surface: all linear, squared and interaction terms of the 4 parameters."""
    return make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), final_step)


def neural_network():
    """Small fully connected network (2 hidden layers of 32 units); the target is standardised before training."""
    network = MLPRegressor(hidden_layer_sizes=(32, 32), solver='lbfgs', alpha=1e-3, max_iter=10_000, random_state=SEED)
    return TransformedTargetRegressor(regressor=network, transformer=StandardScaler())


candidates = {
    'Gaussian process': gaussian_process(),
    'Quadratic response surface': quadratic(RidgeCV(alphas=np.logspace(-6, 2, 30))),
    'Reduced quadratic': quadratic(LassoCV(cv=5, max_iter=100_000, random_state=SEED)),  # lasso drops the weak terms
    'Neural network': neural_network(),
    'Random forest': RandomForestRegressor(n_estimators=300, random_state=SEED),
}

proxies, rows = {}, []
for name, regressor in candidates.items():
    for fluid in MODELLED_FLUIDS:
        started = time.perf_counter()
        proxy = CurveProxy(regressor).fit(inputs(CASES['train']), targets(CASES['train'], fluid))
        proxies[name, fluid] = proxy
        rows.append({'model': name, 'fluid': fluid, 'PCA scores': len(proxy.models),
                     **score(proxy, CASES['validation'], fluid), 'fit time [s]': time.perf_counter() - started})

validation_scores = pd.DataFrame(rows).set_index(['model', 'fluid'])
print('Validation cases 71-85:')
print(validation_scores.round(4).to_string())

In [ ]:
# 5.2 Model comparison on the validation cases
comparison = validation_scores[ERROR_METRIC].unstack('fluid')[MODELLED_FLUIDS]
comparison = comparison.loc[comparison.mean(axis=1).sort_values(ascending=False).index]  # best model ends up on top
BEST_MODEL = comparison.mean(axis=1).idxmin()
if BEST_MODEL != 'Gaussian process':
    raise RuntimeError('The GP-specific tuning workflow must be revisited: a different family won validation.')

positions, bar_height = np.arange(len(comparison)), 0.36
fig, ax = plt.subplots(figsize=(9, 4.4))
for offset, fluid in zip((bar_height / 2, -bar_height / 2), MODELLED_FLUIDS):
    bars = ax.barh(positions + offset, comparison[fluid], height=bar_height - 0.04, color=COLOURS[fluid], label=f'Cumulative {fluid} curve')
    ax.bar_label(bars, fmt='{:.2f}%', padding=4)
ax.set_yticks(positions, comparison.index)
ax.set_xlim(0, comparison.to_numpy().max() * 1.15)
ax.set_xlabel('Error on the 15 validation cases [% of the 2008 value]; shorter is better')
ax.grid(axis='y', visible=False)
finish(fig, f'The {BEST_MODEL} has the lowest error of the {len(comparison)} models by mean oil/water validation error', 'proxy_model_comparison')

In [ ]:
# 5.3 Fine-tuning: 5-fold cross-validation on the training cases only (validation and test cases stay unseen)
from sklearn.model_selection import KFold

# The optimiser sometimes stops early on a flat optimum; the restarts inside each fit cover that case
warnings.filterwarnings('ignore', message='lbfgs failed to converge', category=ConvergenceWarning)

WIDE_RANGE_PARAMS = (PARAMS.index('perm_mult'), PARAMS.index('aquifer_pv'))  # candidates for a log scale
STARTING_SETTINGS = {'kernel': 'RBF', 'variance': 0.9999, 'log_columns': ()}
SETTINGS = [{'kernel': kernel, 'variance': variance, 'log_columns': log_columns}
            for kernel in ('RBF', 'Matern 5/2', 'Matern 3/2')  # how smooth the response is assumed to be, smoothest first
            for variance in (0.999, 0.9999, 0.99999)           # how much of each curve the PCA scores must keep
            for log_columns in ((), WIDE_RANGE_PARAMS)]


def build_proxy(kernel, variance, log_columns):
    return CurveProxy(gaussian_process(kernel), variance=variance, log_columns=log_columns)


def cross_validated_error(settings, n_folds=5):
    """Curve error on held-out training cases, averaged over the folds and over oil and water."""
    errors = []
    for fit_rows, check_rows in KFold(n_folds, shuffle=True, random_state=SEED).split(CASES['train']):
        fit_cases, check_cases = CASES['train'][fit_rows], CASES['train'][check_rows]
        for fluid in MODELLED_FLUIDS:
            fold_proxy = build_proxy(**settings).fit(inputs(fit_cases), targets(fit_cases, fluid))
            errors.append(score(fold_proxy, check_cases, fluid)[ERROR_METRIC])
    return np.mean(errors)


tuning = pd.DataFrame(SETTINGS)
tuning['variance'] = tuning['variance'].map('{:.3%}'.format)
tuning['inputs'] = np.where(tuning.pop('log_columns').astype(bool), 'log permeability and aquifer', 'as given')
tuning['cross-validated error [%]'] = [cross_validated_error(settings) for settings in SETTINGS]
tuning = tuning.sort_values('cross-validated error [%]').rename(columns={'variance': 'curve variance kept by PCA'})
best_settings = SETTINGS[tuning.index[0]]
starting_error, best_error = tuning.loc[SETTINGS.index(STARTING_SETTINGS), 'cross-validated error [%]'], tuning['cross-validated error [%]'].iloc[0]

print('Cross-validated curve error for each setting, best first:')
print(tuning.to_string(index=False, float_format='{:.4f}'.format))
print('\nbest error of each kernel:')
print(tuning.groupby('kernel')['cross-validated error [%]'].min().sort_values().to_string(float_format='{:.4f}'.format))
print(f'\nstarting settings: {starting_error:.4f}%   best settings: {best_error:.4f}%')

labels = tuning['kernel'] + '  |  ' + tuning['curve variance kept by PCA'] + '  |  ' + tuning['inputs']
group = np.select([tuning.index == tuning.index[0], tuning.index == SETTINGS.index(STARTING_SETTINGS)],
                  ['Chosen settings', 'Starting settings'], 'Other settings tried')
group_colours = {'Chosen settings': COLOURS['highlight'], 'Starting settings': COLOURS['neutral'], 'Other settings tried': COLOURS['simulated']}

fig, ax = plt.subplots(figsize=(10, 6.8))
for name, colour in group_colours.items():
    rows = np.flatnonzero(group == name)
    bars = ax.barh(rows, tuning['cross-validated error [%]'].iloc[rows], height=0.65, color=colour, label=name)
    ax.bar_label(bars, fmt='{:.3f}%', padding=4)
ax.set_yticks(range(len(labels)), labels)
ax.invert_yaxis()
ax.set_xlim(0, tuning['cross-validated error [%]'].max() * 1.15)
ax.set_xlabel('Cross-validated error [% of the 2008 value]; shorter is better')
ax.set_title('Each row: kernel  |  share of the curve kept by PCA  |  how permeability and aquifer size enter the model')
ax.grid(axis='y', visible=False)
finish(fig, f'Fine-tuning halves the proxy error, from {starting_error:.2f}% to {best_error:.2f}%', 'proxy_fine_tuning')

In [ ]:
# 5.4 Final check of the tuned model on the validation cases (the test responses remain unseen)
proxy = {fluid: build_proxy(**best_settings).fit(inputs(CASES['train']), targets(CASES['train'], fluid)) for fluid in MODELLED_FLUIDS}

development_scores = pd.DataFrame({fluid: score(proxy[fluid], CASES['validation'], fluid) for fluid in MODELLED_FLUIDS}).T
print('Tuned model, validation cases 71-85:')
print(development_scores.round(4).to_string())

# Speed: one proxy call for many parameter sets
rng = np.random.default_rng(SEED)
random_sets = rng.uniform(params[PARAMS].min(), params[PARAMS].max(), size=(10_000, len(PARAMS)))
started = time.perf_counter()
for fluid in MODELLED_FLUIDS:
    proxy[fluid].predict(random_sets)
print(f'\nproxy time for 10,000 parameter sets: {time.perf_counter() - started:.2f} s')

# Which parameters does the proxy rely on? A short length scale means the output changes quickly with that parameter
relevance = pd.DataFrame({fluid: 1 / proxy[fluid].models[0].kernel_.k1.k2.length_scale for fluid in MODELLED_FLUIDS}, index=PARAMS)
print('\nparameter relevance (1 / GP length scale) for the main PCA score:')
print((relevance / relevance.sum()).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, fluid in zip(axes, MODELLED_FLUIDS):
    actual = targets(CASES['validation'], fluid)[:, -1] / 1e6
    predicted = proxy[fluid].predict(inputs(CASES['validation']))[:, -1] / 1e6
    limits = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]
    ax.plot(limits, limits, color=COLOURS['muted'], linewidth=1, label='Perfect agreement')
    ax.scatter(actual, predicted, s=45, color=COLOURS[fluid], edgecolor='white', zorder=3, label=f'One validation case, {fluid}')
    ax.set_xlabel('Simulator [million STB]')
    ax.set_ylabel('Proxy [million STB]')
    ax.set_title(f'Cumulative {fluid} in 2008')
finish(fig, f'On the {len(CASES["validation"])} validation cases the proxy misses the 2008 total by {development_scores["2008 value MAPE [%]"].max():.2f}% or less on average',
       'proxy_tuned_validation_accuracy')

In [ ]:
# 5.5 Overfitting check: is the error on unseen cases close to the error on the cases the model was fitted on?
# A model that memorises its training cases has a tiny training error and a much larger error on unseen cases.
def mean_error(fitted, cases):
    """Curve error of a {fluid: proxy} pair on the given cases, averaged over oil and water."""
    return np.mean([score(fitted[fluid], cases, fluid)[ERROR_METRIC] for fluid in MODELLED_FLUIDS])


fitted_models = {name: {fluid: proxies[name, fluid] for fluid in MODELLED_FLUIDS} for name in candidates}
fitted_models[f'{BEST_MODEL}, fine-tuned'] = proxy  # every model here was fitted on the 70 training cases only

overfitting = pd.DataFrame({
    name: {'training cases [%]': mean_error(fitted, CASES['train']), 'unseen validation cases [%]': mean_error(fitted, CASES['validation'])}
    for name, fitted in fitted_models.items()
}).T
overfitting['unseen / training'] = overfitting['unseen validation cases [%]'] / overfitting['training cases [%]']
overfitting = overfitting.sort_values('unseen validation cases [%]')
chosen = overfitting.index[0]
case_spread = np.mean([100 * cum[fluid][CASES['train']].iloc[-1].std() / cum[fluid][CASES['train']].iloc[-1].mean() for fluid in MODELLED_FLUIDS])

print('Curve error [% of the 2008 value], averaged over oil and water:')
print(overfitting.round(3).to_string())
print("\nTest responses remain unseen until the final model is frozen in section 5.7.")
print(f'for scale, the 2008 values of the cases themselves differ by about {case_spread:.0f}% (standard deviation)')

positions, bar_height = np.arange(len(overfitting)), 0.36
fig, ax = plt.subplots(figsize=(9, 4.6))
for offset, column, colour in ((-bar_height / 2, 'training cases [%]', COLOURS['simulated']),
                               (bar_height / 2, 'unseen validation cases [%]', COLOURS['highlight'])):
    bars = ax.barh(positions + offset, overfitting[column], height=bar_height - 0.04, color=colour,
                   label='Error on the cases it was fitted on' if 'training' in column else 'Error on cases it has never seen')
    ax.bar_label(bars, fmt='{:.2f}%', padding=4)
ax.set_yticks(positions, overfitting.index)
ax.invert_yaxis()
ax.set_xlim(0, overfitting['unseen validation cases [%]'].max() * 1.15)
ax.set_xlabel('Curve error [% of the 2008 value]; compare generalization gaps across models')
ax.grid(axis='y', visible=False)
finish(fig, f'Training versus validation error: {overfitting.loc[chosen, "training cases [%]"]:.2f}% error on training cases, '
            f'{overfitting.loc[chosen, "unseen validation cases [%]"]:.2f}% on unseen cases; the neural network is '
            f'{overfitting.loc["Neural network", "unseen / training"]:.0f} times worse on unseen cases', 'proxy_overfitting_check')

In [ ]:
# 5.6 Are 70 training cases enough? Retrain on fewer cases and score on the same validation cases
learning_rows = []
for n_cases in (10, 20, 40, 70):
    for fluid in MODELLED_FLUIDS:
        subset = CASES['train'][:n_cases]
        smaller_proxy = build_proxy(**best_settings).fit(inputs(subset), targets(subset, fluid))
        learning_rows.append({'training cases': n_cases, 'fluid': fluid,
                              **score(smaller_proxy, CASES['validation'], fluid)})

learning_curve = pd.DataFrame(learning_rows).pivot(index='training cases', columns='fluid', values='2008 value MAPE [%]')
print('Validation error of the 2008 cumulative value [%]:')
print(learning_curve.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4.2))
for fluid in MODELLED_FLUIDS:
    ax.plot(learning_curve.index, learning_curve[fluid], marker='o', linewidth=2, color=COLOURS[fluid], label=f'Cumulative {fluid}')
ax.set_xticks(learning_curve.index)
ax.set_xlabel('Number of training cases used')
ax.set_ylabel('Error on the validation cases [%]')
ax.set_ylim(bottom=0)
finish(fig, f'Even 10 training cases give an error of {learning_curve.loc[10].max():.2f}% or less; 70 bring it down to '
            f'{learning_curve.loc[70].max():.2f}%', 'proxy_learning_curve')

In [ ]:
# 5.7 Final proxy: refit on the training and validation cases together and save it. The test cases are never used for fitting
import joblib

# MODEL_DIR comes from the project root in setup.

fit_cases = CASES['train'].append(CASES['validation'])
proxy = {fluid: build_proxy(**best_settings).fit(inputs(fit_cases), targets(fit_cases, fluid)) for fluid in MODELLED_FLUIDS}

final_test_scores = pd.DataFrame({fluid: score(proxy[fluid], CASES['test'], fluid) for fluid in MODELLED_FLUIDS}).T
print(f'Test cases 86-100, proxy refitted on {len(fit_cases)} cases:')
print(final_test_scores.round(4).to_string())
test_scores = final_test_scores  # compatibility for the baseline report helpers
final_test_scores.to_csv(OUTPUT_DIR / 'proxy_test_scores.csv')

# Loading this file needs the CurveProxy class from this notebook
joblib.dump({'proxy': proxy, 'parameters': PARAMS, 'dates': dates, 'settings': best_settings, 'package_versions': PACKAGE_VERSIONS, 'fit_cases': fit_cases.to_list()}, MODEL_DIR / 'curve_proxy.joblib')
print(f'\nsaved to {MODEL_DIR / "curve_proxy.joblib"}')

In [ ]:
# 5.8 The proxy as a diagram
from matplotlib.colors import to_rgba
from matplotlib.patches import FancyBboxPatch

n_scores = len(proxy['oil'].models)
log_names = ' and\n'.join(PARAM_LABELS[PARAMS[column]].split()[0].lower() for column in best_settings['log_columns'])
STEPS = [  # (heading, detail, learned from the training cases?)
    ('1. INPUT', '4 parameters\nof one case', False),
    ('2. SCALE', f'log of {log_names},\nthen standardise' if log_names else 'standardise', False),
    ('3. PREDICT', f'{n_scores} Gaussian processes,\none per PCA score', True),
    ('4. REBUILD', f'{n_scores} PCA scores back to\na {len(dates)}-point curve', True),
    ('5. OUTPUT', 'cumulative curve\n1998-2008', False),
]
BOX_COLOURS = {True: COLOURS['highlight'], False: COLOURS['simulated']}

fig, ax = plt.subplots(figsize=(13, 2.8))
ax.axis('off')
ax.set_xlim(0, len(STEPS))
ax.set_ylim(0, 1)
for position, (heading, detail, learned) in enumerate(STEPS):
    ax.add_patch(FancyBboxPatch((position + 0.1, 0.08), 0.74, 0.84, boxstyle='round,pad=0.02',
                                facecolor=to_rgba(BOX_COLOURS[learned], 0.3), edgecolor=BOX_COLOURS[learned], linewidth=1.5))
    ax.text(position + 0.47, 0.76, heading, ha='center', va='center', fontweight='bold')
    ax.text(position + 0.47, 0.38, detail, ha='center', va='center', fontsize=9.5, linespacing=1.4)
    if position:
        ax.annotate('', xy=(position + 0.07, 0.5), xytext=(position - 0.13, 0.5),
                    arrowprops={'arrowstyle': '->', 'color': COLOURS['neutral'], 'linewidth': 1.5})
for learned, label in ((True, 'Learned from the simulated cases'), (False, 'Fixed step')):
    ax.plot([], [], marker='s', linestyle='', markersize=12, markerfacecolor=to_rgba(BOX_COLOURS[learned], 0.3),
            markeredgecolor=BOX_COLOURS[learned], label=label)
finish(fig, 'The proxy turns 4 parameters into a production curve in five steps; one proxy for oil, one for water, gas is a fixed share of oil',
       'proxy_architecture')

## 6. History matching

The objective averages four normalized RMSE terms: cumulative oil, cumulative water,
quarterly oil rate, and quarterly water rate. Each is divided by its observed mean.
Gas is derived from oil and therefore is not counted twice in the objective.

**6.1** retains the ten best actual simulator cases as a benchmark. **6.2** searches
the bounded parameter space with the final proxy, combines the optimum, Sobol samples,
and provided parameter sets, and selects ten distinct realizations by proxy mismatch.
The original cases and optimized realizations have separate identities and curves.
**6.3–6.5** retain benchmark diagnostics; **6.6** compares those with optimized histories,
including the last two years and terminal rates. A nonzero residual is reported honestly:
finding a minimum does not establish an exact match or a simulator-verified optimum.

In [ ]:
# 6.1 Mismatch score and ranking of the simulated cases
QUARTER_DAYS = np.diff(days_since(dates, dates[0]))
N_SELECTED = 10


def normalised_rmse(simulated, measured):
    """RMSE over time divided by the mean observed value; `simulated` may hold many cases (cases x dates)."""
    return np.sqrt(((simulated - measured) ** 2).mean(axis=-1)) / measured.mean()


def mismatch(oil_cum, water_cum):
    """Mismatch to the observed history in percent, for one curve or a (cases x dates) array of curves."""
    terms = []
    for fluid, simulated in (('oil', oil_cum), ('water', water_cum)):
        measured = obs_cum[fluid].to_numpy()
        terms.append(normalised_rmse(simulated, measured))
        terms.append(normalised_rmse(np.diff(simulated, axis=-1) / QUARTER_DAYS, np.diff(measured) / QUARTER_DAYS))
    return 100 * np.mean(terms, axis=0)


all_cases = params.index
ranking = params.assign(mismatch=mismatch(targets(all_cases, 'oil'), targets(all_cases, 'water'))).sort_values('mismatch')
selected_cases = ranking.index[:N_SELECTED]

print(f'mismatch over the 100 cases: best {ranking["mismatch"].min():.2f}%, '
      f'median {ranking["mismatch"].median():.2f}%, worst {ranking["mismatch"].max():.2f}%')
print(f'\nthe {N_SELECTED} best-matched cases:')
print(ranking.head(N_SELECTED).round(3).to_string())

In [ ]:
# 6.2 Search and select the ten optimized realizations used by the deliverable
from scipy.optimize import differential_evolution
from scipy.stats import qmc

lower, upper = params[PARAMS].min().to_numpy(), params[PARAMS].max().to_numpy()


def proxy_mismatch(parameter_sets, batch_size=4096):
    parameter_sets = np.atleast_2d(parameter_sets)
    return np.concatenate([
        mismatch(proxy['oil'].predict(batch), proxy['water'].predict(batch))
        for batch in (parameter_sets[start:start + batch_size]
                      for start in range(0, len(parameter_sets), batch_size))
    ])


def select_realizations(candidate_parameters, candidate_scores, candidate_sources, count=10):
    """Sort deterministically, then retain distinct sets at 8 decimal places in normalized space."""
    candidates = pd.DataFrame(candidate_parameters, columns=PARAMS)
    candidates['mismatch'] = np.asarray(candidate_scores)
    candidates['candidate_source'] = candidate_sources
    valid = (np.isfinite(candidates[PARAMS + ['mismatch']]).all(axis=1)
             & candidates[PARAMS].ge(lower - 1e-12).all(axis=1)
             & candidates[PARAMS].le(upper + 1e-12).all(axis=1))
    candidates = candidates.loc[valid].sort_values(['mismatch', *PARAMS], kind='stable')
    normalized = ((candidates[PARAMS] - lower) / (upper - lower)).round(8)
    chosen = candidates.loc[~normalized.duplicated()].head(count).copy()
    if len(chosen) != count:
        raise ValueError(f'Only {len(chosen)} valid, distinct realizations; {count} required.')
    chosen.index = pd.Index([f'optimized_{rank:02d}' for rank in range(1, count + 1)], name='realization_id')
    chosen['source'] = 'proxy_optimized'
    return chosen


search = differential_evolution(lambda x: proxy_mismatch(x)[0], list(zip(lower, upper)), seed=SEED, tol=1e-8)
if not np.isfinite(search.fun):
    raise RuntimeError('The optimizer did not produce a finite history-match objective.')
best_fit = pd.Series(search.x, index=PARAMS)
print(f'Best optimizer parameter set, proxy mismatch {search.fun:.3f}% (converged: {search.success}):')
print(best_fit.round(6).to_string())

samples = qmc.scale(qmc.Sobol(d=len(PARAMS), seed=SEED).random_base2(17), lower, upper)
sample_mismatch = proxy_mismatch(samples)
threshold = ranking['mismatch'].iloc[N_SELECTED - 1]
accepted = pd.DataFrame(samples[sample_mismatch <= threshold], columns=PARAMS)
if accepted.empty:
    raise RuntimeError('No sampled parameter sets meet the benchmark mismatch threshold.')
constraint = pd.DataFrame({'full min': lower, 'full max': upper,
                           'matched min': accepted.min().to_numpy(), 'matched max': accepted.max().to_numpy()}, index=PARAMS)
constraint['range kept [%]'] = 100 * (constraint['matched max'] - constraint['matched min']) / (upper - lower)
print(f'\n{len(accepted):,} of {len(samples):,} sampled sets match within {threshold:.2f}%.')
print(constraint.round(3).to_string())

provided_proxy_scores = proxy_mismatch(inputs(params.index))
candidate_parameters = np.vstack([search.x, samples, inputs(params.index)])
candidate_scores = np.concatenate([[search.fun], sample_mismatch, provided_proxy_scores])
candidate_sources = ['differential_evolution'] + ['sobol'] * len(samples) + ['provided_parameters'] * len(params)
optimized_cases = select_realizations(candidate_parameters, candidate_scores, candidate_sources, N_SELECTED)
assert optimized_cases['mismatch'].iloc[0] <= provided_proxy_scores.min() + 1e-10
assert optimized_cases['mismatch'].iloc[0] <= search.fun + 1e-10
FIELD_GOR = float(obs_cum['gas'].iloc[-1] / obs_cum['oil'].iloc[-1])  # MSCF per STB
optimized_cum = pd.concat({
    fluid: pd.DataFrame(proxy[fluid].predict(optimized_cases[PARAMS].to_numpy()).T,
                        index=dates, columns=optimized_cases.index)
    for fluid in MODELLED_FLUIDS
}, axis=1)
optimized_cum = pd.concat([optimized_cum, pd.concat({'gas': optimized_cum['oil'] * FIELD_GOR}, axis=1)], axis=1)
print('\nTen optimized realizations (proxy predictions, not simulator runs):')
print(optimized_cases.round(6).to_string())
print(f'Best provided simulator mismatch: {ranking["mismatch"].min():.3f}%; '
      f'best provided parameters under the same proxy: {provided_proxy_scores.min():.3f}%.')

In [ ]:
# 6.3 What the match looks like
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, fluid in zip(axes, MODELLED_FLUIDS):
    ax.plot(rate[fluid], color=COLOURS['simulated'], linewidth=0.5)
    ax.plot(rate[fluid][selected_cases], color=COLOURS['highlight'], linewidth=1.2)
    ax.plot(obs_rate[fluid], color=COLOURS['observed'], linewidth=2)
    ax.set_title(f'{fluid.capitalize()} rate [STB per day]')
axes[0].plot([], [], color=COLOURS['simulated'], label='The other simulated cases')
axes[0].plot([], [], color=COLOURS['highlight'], linewidth=1.2, label=f'The {N_SELECTED} best-matched cases')
axes[0].plot([], [], color=COLOURS['observed'], linewidth=2, label='Real field')
finish(fig, f'The {N_SELECTED} best-matched cases follow the real field closely until 2005 and drift apart after that', 'history_match_selected_cases')

# How far the selected cases are from the observed field at the end of the history
end_of_history = pd.DataFrame({
    'observed': {f'{fluid} rate, last quarter [STB/d]': obs_rate[fluid].iloc[-1] for fluid in MODELLED_FLUIDS}
                | {f'cumulative {fluid} [million STB]': obs_cum[fluid].iloc[-1] / 1e6 for fluid in MODELLED_FLUIDS},
    'selected cases, mean': {f'{fluid} rate, last quarter [STB/d]': rate[fluid][selected_cases].iloc[-1].mean() for fluid in MODELLED_FLUIDS}
                            | {f'cumulative {fluid} [million STB]': cum[fluid][selected_cases].iloc[-1].mean() / 1e6 for fluid in MODELLED_FLUIDS},
})
end_of_history['difference [%]'] = 100 * (end_of_history['selected cases, mean'] / end_of_history['observed'] - 1)
print(end_of_history.round(2).to_string())

In [ ]:
# 6.4 Does the gap to the real field change over the years? Yearly production of the selected cases against the observed
MIN_SHARE_OF_AVERAGE = 0.05  # skip years in which the field produced almost none of the fluid, where a percentage means little
STEADY_UNTIL = 2005
year_ends = dates[::4]
yearly_gap = {}
for fluid in MODELLED_FLUIDS:
    observed_volume = obs_cum[fluid].loc[year_ends].diff().iloc[1:]
    simulated_volume = cum[fluid][selected_cases].loc[year_ends].diff().iloc[1:]
    gap = 100 * (simulated_volume.div(observed_volume, axis=0) - 1)
    gap.index = gap.index.year - 1  # the volume between two Januaries belongs to the earlier year
    yearly_gap[fluid] = gap[(observed_volume >= MIN_SHARE_OF_AVERAGE * observed_volume.mean()).to_numpy()]

mean_gap = pd.DataFrame({fluid: gap.mean(axis=1) for fluid, gap in yearly_gap.items()})
print('Yearly production of the selected cases compared with the real field, mean difference [%]:')
print(mean_gap.round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 4.6))
for fluid, gap in yearly_gap.items():
    ax.fill_between(gap.index, gap.min(axis=1), gap.max(axis=1), color=COLOURS[fluid], alpha=0.15, linewidth=0)
    ax.plot(gap.index, gap.mean(axis=1), marker='o', linewidth=2, color=COLOURS[fluid], label=f'{fluid.capitalize()} produced in the year')
ax.axhline(0, color=COLOURS['observed'], linewidth=1.5, label='Real field (0% = perfect match)')
ax.set_xticks(mean_gap.index)
ax.set_ylabel('Selected cases above (+) or below (-)\nthe real field [%]')
ax.set_title(f'Line: average of the {N_SELECTED} selected cases. Shaded band: their range.')
finish(fig, f'Oil stays within {mean_gap.loc[:STEADY_UNTIL, "oil"].abs().max():.1f}% of the real field until {STEADY_UNTIL}, '
            f'then runs {mean_gap["oil"].iloc[-1]:.0f}% too high by {mean_gap.index[-1]}', 'history_match_gap_by_year')

In [ ]:
# 6.5 Which parameters does the history pin down? Range of the matching parameter sets as a share of the full range
kept = constraint.sort_values('range kept [%]', ascending=False)
start = 100 * (kept['matched min'] - kept['full min']) / (kept['full max'] - kept['full min'])
labels = [PARAM_LABELS[param] for param in kept.index]

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.barh(labels, 100, height=0.5, color=COLOURS['simulated'], alpha=0.5, label='Full range of the parameter')
ax.barh(labels, kept['range kept [%]'], left=start, height=0.5, color=COLOURS['highlight'], label='Values that still match the history')
for position, (param, row) in enumerate(kept.iterrows()):
    ax.text(103, position, f'{row["range kept [%]"]:.0f}% of the range still matches\n'
                           f'({row["matched min"]:.3g} to {row["matched max"]:.3g})', va='center')
ax.set_xlim(0, 150)
ax.set_xticks([0, 50, 100], ['lowest value', 'middle', 'highest value'])
ax.grid(visible=False)
finish(fig, f'The history narrows porosity to {constraint.loc["poro_mult", "range kept [%]"]:.0f}% of its range '
            f'and leaves the other three parameters open', 'history_match_parameter_constraint')

### 6.6 Provided versus optimized curves and recent residuals

In [ ]:
# 6.6 Compare provided simulator cases with the selected optimized proxy histories
optimized_rate = quarterly_rates(optimized_cum)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))
for ax, fluid in zip(axes, MODELLED_FLUIDS):
    ax.plot(rate[fluid][selected_cases], color=COLOURS['simulated'], linewidth=0.8)
    ax.plot(optimized_rate[fluid], color=COLOURS['highlight'], linewidth=1)
    ax.plot(obs_rate[fluid], color=COLOURS['observed'], linewidth=2)
    ax.set_title(f'{fluid.capitalize()} rate [STB/day]')
axes[0].plot([], [], color=COLOURS['simulated'], label='Provided simulator benchmark')
axes[0].plot([], [], color=COLOURS['highlight'], label='Optimized proxy histories')
axes[0].plot([], [], color=COLOURS['observed'], label='Observed field')
finish(fig, f'History-match objective: best simulator {ranking["mismatch"].min():.2f}%; '
            f'best optimized proxy {optimized_cases["mismatch"].min():.2f}%', 'history_match_optimized_cases')
recent = dates > pd.Timestamp('2006-01-01')  # eight quarters ending April 2006–January 2008
recent_residuals = pd.DataFrame({
    f'{fluid} {measure}': {
        'provided benchmark': 100 * (rate[fluid][selected_cases].mean(axis=1).loc[recent].mean() / obs_rate[fluid].loc[recent].mean() - 1)
                              if measure == 'recent rate bias [%]' else
                              100 * (rate[fluid][selected_cases].iloc[-1].mean() / obs_rate[fluid].iloc[-1] - 1),
        'optimized proxy': 100 * (optimized_rate[fluid].mean(axis=1).loc[recent].mean() / obs_rate[fluid].loc[recent].mean() - 1)
                           if measure == 'recent rate bias [%]' else
                           100 * (optimized_rate[fluid].iloc[-1].mean() / obs_rate[fluid].iloc[-1] - 1),
    } for fluid in MODELLED_FLUIDS for measure in ('recent rate bias [%]', 'terminal rate bias [%]')
}).T
print('Residuals before any forecast-state anchoring (positive = model above field):')
print(recent_residuals.round(2).to_string())
recent_residuals.to_csv(OUTPUT_DIR / 'history_match_residuals.csv')
print('Forecast anchoring corrects the starting state; it does not erase these reservoir-model residuals.')

## 7. Production forecast, 2008–2028

No simulated or observed production after January 2008 is supplied. A waterflood WOR
extrapolation forecasts twenty years; it is not a supervised twenty-year proxy prediction.
Each forecast averages its own recent ln(WOR) trend and a pooled type curve normalized
by oil pore volume. Rock properties remain static, gas follows the observed gas/oil ratio,
and no new wells or operating changes are assumed.

The type curve is fitted only on explicit training cases. Section 7.2 uses five-fold
training hindcasts for calibration, validation cases for development diagnostics, and
held-out test cases for the frozen method. Each fit stops at its cutoff date. Hindcasts
using actual future liquid rates isolate WOR error and are **conditional diagnostics**;
flat-liquid hindcasts measure a deployable forecasting assumption.

Section 7.3 retains original simulator forecasts as a benchmark. Sections 7.4–7.6 and
the official export use the optimized realizations. Their WOR trends are translated to
the observed cumulative state and anchored to the last observed oil/water ratio before
future integration. The base case holds the field's last liquid rate constant; decline
scenarios and conditional uncertainty show the consequences of changing that assumption.

Initial grid pressure is available, but the production workbook contains no observed
pressure time series to match. Normalized oil production / oil pore volume is a type-curve
coordinate, not a claimed stock-tank recovery factor without a formation-volume conversion.

In [ ]:
# 7.1 Forecast method: explicit fit cases, time cutoffs, and optional field-state anchoring
FORECAST_DATES = pd.date_range('2008-04-01', '2028-01-01', freq='QS', name='date')
FORECAST_DAYS = np.diff(days_since(dates[-1:].append(FORECAST_DATES), dates[0]))
OIL_PV = grid_features['oil_pore_volume']
OIL_PV_COEFFICIENT = float((OIL_PV.loc[CASES['train']] / params.loc[CASES['train'], 'poro_mult']).median())
pv_relative_error = ((OIL_PV - OIL_PV_COEFFICIENT * params['poro_mult']).abs() / OIL_PV).max()
assert pv_relative_error < 1e-6, 'The porosity–oil-pore-volume relationship must be revalidated for this dataset.'
optimized_oil_pv = OIL_PV_COEFFICIENT * optimized_cases['poro_mult']
FIT_QUARTERS, ANCHOR_QUARTERS = 8, 4
MIN_LOG_WOR = np.log(0.02 / 0.98)
TYPE_CURVE_FITS = []


def quarterly_log_wor(oil_cum, water_cum):
    oil, water = np.diff(oil_cum), np.diff(water_cum)
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.log(water / oil), (oil_cum[1:] + oil_cum[:-1]) / 2


def build_type_curve(last, fitting_cases, bin_width=0.01, tail_width=0.04, min_points=5):
    """Fit on named cases and dates up to `last`; never infer the fitting pool from all cases."""
    fitting_cases = pd.Index(fitting_cases)
    if fitting_cases.empty or not fitting_cases.is_unique or not 1 <= last < len(dates):
        raise ValueError('A type curve requires unique fitting cases and a valid historical cutoff.')
    points = []
    for case in fitting_cases:
        log_wor, mid_cum = quarterly_log_wor(cum['oil'][case].to_numpy()[:last + 1],
                                            cum['water'][case].to_numpy()[:last + 1])
        keep = np.isfinite(log_wor) & (log_wor > MIN_LOG_WOR)
        points.append(pd.DataFrame({'recovery': mid_cum[keep] / OIL_PV[case], 'log_wor': log_wor[keep]}))
    points = pd.concat(points, ignore_index=True)
    bins = points.groupby(np.floor(points['recovery'] / bin_width)).agg(
        recovery=('recovery', 'mean'), log_wor=('log_wor', 'median'), n=('log_wor', 'size'))
    bins = bins[bins['n'] >= min_points]
    if len(bins) < 2:
        raise ValueError('Too few supported bins to fit a WOR type curve at this cutoff.')
    recovery, log_wor = bins['recovery'].to_numpy(), np.maximum.accumulate(bins['log_wor'].to_numpy())
    tail = recovery >= recovery[-1] - tail_width
    if tail.sum() < 2:
        tail[-2:] = True
    slope = max(0.0, float(np.polyfit(recovery[tail], log_wor[tail], 1)[0]))
    TYPE_CURVE_FITS.append({'fit_cases': tuple(int(c) for c in fitting_cases), 'fit_end': dates[last]})
    return recovery, log_wor, slope


def type_curve_value(recovery, curve):
    x, y, tail_slope = curve
    return np.interp(recovery, x, y) if recovery <= x[-1] else y[-1] + tail_slope * (recovery - x[-1])


def own_trend(oil_cum, water_cum):
    log_wor, mid_cum = quarterly_log_wor(oil_cum[-FIT_QUARTERS - 1:], water_cum[-FIT_QUARTERS - 1:])
    keep = np.isfinite(log_wor) & np.isfinite(mid_cum)
    if keep.sum() < 2 or np.ptp(mid_cum[keep]) == 0:
        raise ValueError('The own WOR trend needs two finite, distinct recent oil-production coordinates.')
    slope, intercept = np.polyfit(mid_cum[keep], log_wor[keep], 1)
    return lambda cumulative: intercept + slope * cumulative


def type_curve_trend(oil_cum, water_cum, oil_pv, curve):
    if not np.isfinite(oil_pv) or oil_pv <= 0:
        raise ValueError('Oil pore volume must be finite and positive.')
    log_wor, mid_cum = quarterly_log_wor(oil_cum[-ANCHOR_QUARTERS - 1:], water_cum[-ANCHOR_QUARTERS - 1:])
    keep = np.isfinite(log_wor) & np.isfinite(mid_cum)
    if not keep.any():
        raise ValueError('No finite recent WOR values are available to anchor the type curve.')
    shift = np.mean([value - type_curve_value(c / oil_pv, curve)
                     for value, c in zip(log_wor[keep], mid_cum[keep])])
    return lambda cumulative: type_curve_value(cumulative / oil_pv, curve) + shift


def anchor_trend(trend, history_oil_end, field_oil_end, field_log_wor):
    """Preserve the trend shape, move its origin to the field, and match WOR at the boundary."""
    offset = history_oil_end - field_oil_end
    shift = field_log_wor - trend(history_oil_end)
    return lambda cumulative: trend(cumulative + offset) + shift


def integrate_oil(oil_start, log_wor_at, liquid_rates, days):
    liquid_rates, days = np.asarray(liquid_rates), np.asarray(days)
    if (liquid_rates.ndim != 1 or liquid_rates.shape != days.shape or not np.isfinite(oil_start)
            or oil_start < 0 or not np.isfinite(liquid_rates).all() or (liquid_rates < 0).any()
            or not np.isfinite(days).all() or (days <= 0).any()):
        raise ValueError('Integration requires a nonnegative start and aligned finite rates/days with positive days.')
    cumulative, path = oil_start, []
    for liquid, step in zip(liquid_rates, days):
        oil_rate = liquid / (1 + np.exp(np.clip(log_wor_at(cumulative), -700, 700)))
        for _ in range(4):
            oil_rate = liquid / (1 + np.exp(np.clip(log_wor_at(cumulative + 0.5 * oil_rate * step), -700, 700)))
        cumulative += oil_rate * step
        path.append(cumulative)
    return np.asarray(path)


def forecast_oil(oil_cum, water_cum, oil_pv, curve, liquid_rates, days, *, initial_oil=None, anchor_log_wor=None):
    """Forecast from a simulator state, or from an explicitly anchored observed field state."""
    initial_oil = oil_cum[-1] if initial_oil is None else initial_oil
    trends = {'own trend': own_trend(oil_cum, water_cum),
              'type curve': type_curve_trend(oil_cum, water_cum, oil_pv, curve)}
    if anchor_log_wor is not None:
        if not np.isfinite(anchor_log_wor):
            raise ValueError('The field WOR anchor must be finite.')
        trends = {name: anchor_trend(trend, oil_cum[-1], initial_oil, anchor_log_wor)
                  for name, trend in trends.items()}
    paths = {name: integrate_oil(initial_oil, trend, liquid_rates, days) for name, trend in trends.items()}
    paths['average'] = (paths['own trend'] + paths['type curve']) / 2
    return paths


def last_liquid_rate(oil_cum, water_cum, days):
    return (oil_cum[-1] - oil_cum[-2] + water_cum[-1] - water_cum[-2]) / days


print(f'Oil pore volume = {OIL_PV_COEFFICIENT:.6f} × porosity multiplier; maximum relative residual {pv_relative_error:.2e}.')

In [ ]:
# 7.2 Held-out hindcasts; only training-fold errors enter uncertainty calibration
HINDCAST_CUTS = ('2003-01-01', '2004-01-01', '2005-01-01', '2006-01-01')


def hindcast(cut_date, cases, fitting_cases, partition, fold=None):
    cases, fitting_cases = pd.Index(cases), pd.Index(fitting_cases)
    if len(cases.intersection(fitting_cases)):
        raise ValueError('Hindcast evaluation cases must not enter the fitted type curve.')
    cut = dates.get_loc(pd.Timestamp(cut_date))
    curve, days = build_type_curve(cut, fitting_cases), QUARTER_DAYS[cut:]
    records = []
    for case in cases:
        oil, water = cum['oil'][case].to_numpy(), cum['water'][case].to_numpy()
        produced = oil[-1] - oil[cut]
        if produced <= 0:
            raise ValueError(f'Case {case}: no post-cut oil volume for percentage-error evaluation.')
        log_anchor = np.log((water[cut] - water[cut - 1]) / (oil[cut] - oil[cut - 1]))
        liquid_options = {
            'actual liquid rate': (np.diff(oil[cut:]) + np.diff(water[cut:])) / days,
            'flat liquid rate': np.full(len(days), last_liquid_rate(oil[:cut + 1], water[:cut + 1], QUARTER_DAYS[cut - 1])),
        }
        for liquid_name, liquid_rates in liquid_options.items():
            paths = forecast_oil(oil[:cut + 1], water[:cut + 1], OIL_PV[case], curve, liquid_rates, days,
                                 initial_oil=oil[cut], anchor_log_wor=log_anchor)
            for method, path in paths.items():
                records.append({'cut': cut_date, 'years ahead': (len(dates) - 1 - cut) / 4,
                                'case': case, 'liquid': liquid_name, 'method': method,
                                'error [%]': 100 * ((path[-1] - oil[cut]) / produced - 1),
                                'partition': partition, 'fold': fold,
                                'fit_cases': ','.join(map(str, fitting_cases)),
                                'fit_end': dates[cut], 'future_liquid_used': liquid_name == 'actual liquid rate'})
    return pd.DataFrame(records)


parts = []
for fold, (fit_rows, check_rows) in enumerate(KFold(5, shuffle=True, random_state=SEED).split(CASES['train']), start=1):
    fold_fit_cases, fold_check_cases = CASES['train'][fit_rows], CASES['train'][check_rows]
    for cut_date in HINDCAST_CUTS:
        parts.append(hindcast(cut_date, fold_check_cases, fold_fit_cases, 'train_cv', fold))
for partition, check_cases, curve_fit_cases in (
    ('validation', CASES['validation'], CASES['train']),
    ('test', CASES['test'], fit_cases),
):
    for cut_date in HINDCAST_CUTS:
        parts.append(hindcast(cut_date, check_cases, curve_fit_cases, partition))
hindcast_results = pd.concat(parts, ignore_index=True)
hindcast_results['abs error [%]'] = hindcast_results['error [%]'].abs()
hindcast_results.to_csv(OUTPUT_DIR / 'hindcast_results.csv', index=False)
print('Actual future liquid = conditional WOR diagnostic; flat liquid = forecast assumption. Neither validates twenty-year coverage.')
for label, subset in hindcast_results.groupby('partition', sort=False):
    table = subset.pivot_table(index=['cut', 'years ahead'], columns=['liquid', 'method'], values='abs error [%]', aggfunc='mean')
    print(f'\nHeld-out mean absolute error of incremental oil [%], {label}:')
    print(table.round(1).to_string())

In [ ]:
# 7.3 Original simulator forecasts to 2028: benchmark only
type_curve = build_type_curve(len(dates) - 1, fit_cases)  # final fit on cases 1–85 only


def forecast_case(case):
    """Cumulative oil, water and gas on the forecast dates, for each WOR trend and their average."""
    oil, water, gas = (cum[fluid][case].to_numpy() for fluid in ('oil', 'water', 'gas'))
    liquid_rate = last_liquid_rate(oil, water, QUARTER_DAYS[-1])
    liquid_cum = liquid_rate * np.cumsum(FORECAST_DAYS)
    oil_paths = forecast_oil(oil, water, OIL_PV[case], type_curve, np.full(len(FORECAST_DAYS), liquid_rate), FORECAST_DAYS)
    return pd.concat({
        method: pd.DataFrame({
            'oil': oil_path,
            'water': water[-1] + liquid_cum - (oil_path - oil[-1]),  # liquid that is not oil is water
            'gas': oil_path * gas[-1] / oil[-1],
        }, index=FORECAST_DATES)
        for method, oil_path in oil_paths.items()
    }, axis=1)


forecasts = {case: forecast_case(case) for case in params.index}

summary = pd.DataFrame({
    case: {
        'mismatch [%]': ranking.loc[case, 'mismatch'],
        'oil 2008 [MMSTB]': cum['oil'][case].iloc[-1] / 1e6,
        'oil 2018 [MMSTB]': forecasts[case]['average', 'oil'].loc['2018-01-01'] / 1e6,
        'oil 2028 [MMSTB]': forecasts[case]['average', 'oil'].iloc[-1] / 1e6,
        'oil 2028, own trend': forecasts[case]['own trend', 'oil'].iloc[-1] / 1e6,
        'oil 2028, type curve': forecasts[case]['type curve', 'oil'].iloc[-1] / 1e6,
        'water 2028 [MMSTB]': forecasts[case]['average', 'water'].iloc[-1] / 1e6,
        'oil rate 2028 [STB/d]': forecasts[case]['average', 'oil'].diff().iloc[-1] / FORECAST_DAYS[-1],
        'recovery 2028': forecasts[case]['average', 'oil'].iloc[-1] / OIL_PV[case],
    } for case in selected_cases
}).T.rename_axis('case')
print(f'type curve covers recovery up to {type_curve[0][-1]:.3f}; beyond that it is a straight-line extrapolation')
print(summary.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, fluid in zip(axes, MODELLED_FLUIDS):
    for case in selected_cases:
        ax.plot(forecasts[case]['average', fluid] / 1e6, color=COLOURS[fluid], linewidth=1)
    ax.plot(obs_cum[fluid] / 1e6, color=COLOURS['observed'], linewidth=2.5)
    ax.axvline(dates[-1], color=COLOURS['muted'], linestyle='--', linewidth=1)
    final = pd.Series({case: forecasts[case]['average', fluid].iloc[-1] / 1e6 for case in selected_cases})
    ax.text(0.03, 0.95, f'2008: {obs_cum[fluid].iloc[-1] / 1e6:.0f} million STB (real field)\n'
                        f'2028: {final.min():.0f} to {final.max():.0f} million STB (forecast)', transform=ax.transAxes, va='top')
    ax.set_title(f'Cumulative {fluid} [million STB]')
axes[0].plot([], [], color=COLOURS['observed'], linewidth=2.5, label='Real field, 1998-2008')
axes[0].plot([], [], color=COLOURS['oil'], label=f'Oil forecast, {N_SELECTED} selected cases')
axes[0].plot([], [], color=COLOURS['water'], label=f'Water forecast, {N_SELECTED} selected cases')
axes[0].plot([], [], color=COLOURS['muted'], linestyle='--', label='End of history')
finish(fig, f'Benchmark only: by 2028 the original cases reach {summary["oil 2028 [MMSTB]"].min():.0f}-{summary["oil 2028 [MMSTB]"].max():.0f} million STB of oil '
            f'and {summary["water 2028 [MMSTB]"].min():.0f}-{summary["water 2028 [MMSTB]"].max():.0f} million STB of water', 'forecast_selected_cases')

### 7.4–7.6 Field scenarios and conditional uncertainty

All official forecast cases start at the observed January 2008 oil, water, and gas totals.
Their WOR trends use proxy-generated optimized histories, translated to the field's
cumulative state and anchored to its latest measured phase split. Section 7.4 varies
liquid decline; section 7.5 draws 20,000 conditional scenarios; section 7.6 compares
the ten base-case forecasts and supplies the charts and numbers used by the export.

| Uncertain input | Sampling assumption |
|---|---|
| Optimized realization | Equal chance among the ten distinct selected parameter sets; not a Bayesian posterior |
| Liquid decline | Triangular between zero and the five-year trend, with the two-year trend as mode |
| WOR trend | Uniform mixture of own trend and type curve |
| Method discrepancy | Observed/predicted factor from training-fold conditional hindcasts only |

P90, P50, and P10 denote the 10th, 50th, and 90th percentiles **under these assumptions**.
They are not empirically established twenty-year probabilities. Marginal oil and water
percentiles are not paired physical scenarios. Hindcast errors span two to five years;
using those factors at twenty years can understate uncertainty. New wells, control changes,
and economic inputs are absent. Close agreement among ten low-mismatch cases is not a
complete measure of forecast uncertainty.

In [ ]:
# 7.4 Scenario test: what if the liquid rate keeps falling instead of staying flat?
FIELD_START = obs_cum.iloc[-1]                                               # the real field's totals in January 2008
obs_liquid_rate = (obs_cum['oil'] + obs_cum['water']).diff().iloc[1:] / QUARTER_DAYS
FORECAST_YEARS = (np.cumsum(FORECAST_DAYS) - FORECAST_DAYS / 2) / 365.25     # middle of each forecast quarter, in years after January 2008
WOR_TRENDS = ['own trend', 'type curve']
PLOT_FROM = pd.Timestamp('2004-01-01')                                       # the forecast figures show the last years of history only


def historical_decline(years):
    """Average yearly fall of the observed liquid rate over the last `years` years, as a fraction."""
    return 1 - (obs_liquid_rate.iloc[-1] / obs_liquid_rate.iloc[-1 - 4 * years]) ** (1 / years)


def declining_liquid(decline):
    """Future liquid rate of the field, starting at the last observed rate and falling by `decline` per year."""
    return obs_liquid_rate.iloc[-1] * (1 - decline) ** FORECAST_YEARS


def new_oil(case, decline):
    """Oil produced after January 2008 under each WOR trend of one matched case, as a (trends x dates) array."""
    oil, water = cum['oil'][case].to_numpy(), cum['water'][case].to_numpy()
    paths = forecast_oil(oil, water, OIL_PV[case], type_curve, declining_liquid(decline), FORECAST_DAYS)
    return np.array([paths[trend] - oil[-1] for trend in WOR_TRENDS])



def optimized_new_oil(realization_id, decline):
    """Incremental field oil from optimized histories, anchored before integration."""
    oil = optimized_cum['oil'][realization_id].to_numpy()
    water = optimized_cum['water'][realization_id].to_numpy()
    paths = forecast_oil(oil, water, optimized_oil_pv[realization_id], type_curve,
                         declining_liquid(decline), FORECAST_DAYS,
                         initial_oil=FIELD_START['oil'], anchor_log_wor=FIELD_LOG_WOR)
    return np.asarray([paths[trend] - FIELD_START['oil'] for trend in WOR_TRENDS])


def field_totals(new_oil_volume, new_liquid_volume):
    """Enforce phase conservation per quarter, supporting both paths and arrays of draws."""
    oil_volume, liquid_volume = np.broadcast_arrays(np.asarray(new_oil_volume, dtype=float),
                                                     np.asarray(new_liquid_volume, dtype=float))
    if not np.isfinite(oil_volume).all() or not np.isfinite(liquid_volume).all():
        raise ValueError('Oil and liquid volumes must be finite.')
    liquid_steps = np.diff(liquid_volume, axis=-1, prepend=np.zeros_like(liquid_volume[..., :1]))
    if (liquid_steps < -1e-8).any():
        raise ValueError('Cumulative liquid production must not decrease.')
    liquid_steps = np.maximum(liquid_steps, 0)
    oil_steps = np.diff(oil_volume, axis=-1, prepend=np.zeros_like(oil_volume[..., :1]))
    oil_steps = np.clip(oil_steps, 0, liquid_steps)
    oil_added = np.cumsum(oil_steps, axis=-1)
    water_added = np.cumsum(liquid_steps - oil_steps, axis=-1)
    return {'oil': FIELD_START['oil'] + oil_added,
            'water': FIELD_START['water'] + water_added,
            'gas': FIELD_START['gas'] + oil_added * FIELD_GOR}


FIELD_LOG_WOR = float(np.log(obs_rate['water'].iloc[-1] / obs_rate['oil'].iloc[-1]))


scenario_declines = pd.Series({
    'Flat liquid rate (official base case)': 0.0,
    'Liquid falls as in the last 2 years': historical_decline(2),
    'Liquid falls as in the last 3 years': historical_decline(3),
    'Liquid falls as in the last 5 years': historical_decline(5),
})

scenarios = {}
for name, decline in scenario_declines.items():
    oil_volume = np.mean([optimized_new_oil(realization, decline).mean(axis=0) for realization in optimized_cases.index], axis=0)  # both trends, all selected cases
    scenarios[name] = pd.DataFrame(field_totals(oil_volume, np.cumsum(declining_liquid(decline) * FORECAST_DAYS)), index=FORECAST_DATES)

scenario_table = pd.DataFrame({
    name: {
        'liquid decline [%/year]': 100 * scenario_declines[name],
        'liquid rate 2028 [STB/d]': declining_liquid(scenario_declines[name])[-1],
        'oil 2018 [MMSTB]': forecast.loc['2018-01-01', 'oil'] / 1e6,
        'oil 2028 [MMSTB]': forecast['oil'].iloc[-1] / 1e6,
        'water 2028 [MMSTB]': forecast['water'].iloc[-1] / 1e6,
        'oil rate 2028 [STB/d]': forecast['oil'].diff().iloc[-1] / FORECAST_DAYS[-1],
    } for name, forecast in scenarios.items()
}).T
scenario_table.loc['Section 7.3 (starts from each case\'s own 2008 totals)'] = {
    'oil 2018 [MMSTB]': summary['oil 2018 [MMSTB]'].mean(), 'oil 2028 [MMSTB]': summary['oil 2028 [MMSTB]'].mean(),
    'water 2028 [MMSTB]': summary['water 2028 [MMSTB]'].mean(), 'oil rate 2028 [STB/d]': summary['oil rate 2028 [STB/d]'].mean()}

print('Observed liquid rate each January [STB/d]:')
print(obs_liquid_rate[obs_liquid_rate.index.month == 1].round(0).astype(int).to_string())
print('\nForecast of the real field, average of the optimized realizations:')
print(scenario_table.round(2).to_string())

base, steepest = scenario_table.index[0], scenario_table.index[len(scenarios) - 1]
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.plot(obs_cum.loc[PLOT_FROM:, 'oil'] / 1e6, color=COLOURS['observed'], linewidth=2.5, label='Real field up to January 2008')
for (name, forecast), style in zip(scenarios.items(), ('-', '--', '-.', ':')):
    is_base = name == base
    ax.plot(forecast['oil'] / 1e6, linestyle=style, linewidth=2.5 if is_base else 1.8, color=COLOURS['oil'] if is_base else COLOURS['neutral'],
            label=f'{name}, {100 * scenario_declines[name]:.1f}% per year: {forecast["oil"].iloc[-1] / 1e6:.1f} million STB in 2028')
ax.axvline(dates[-1], color=COLOURS['muted'], linestyle='--', linewidth=1)
ax.set_ylabel('Cumulative oil [million STB]')
finish(fig, f'If the liquid rate keeps falling at its 5-year pace, oil by 2028 drops from '
            f'{scenario_table.loc[base, "oil 2028 [MMSTB]"]:.0f} to {scenario_table.loc[steepest, "oil 2028 [MMSTB]"]:.0f} million STB',
       'forecast_liquid_scenarios', legend_columns=1)

In [ ]:
# 7.5 Conditional scenario percentiles: repeat the forecast with every uncertain input drawn at random, then read off P90 / P50 / P10
N_DRAWS = 20_000
PERCENTILES = {'P90 (low case)': 10, 'P50 (middle case)': 50, 'P10 (high case)': 90}  # exceedance labels conditional on the specified scenario assumptions
SOURCES = ['matched case', 'liquid rate', 'WOR trend', 'method error']
LIKELY_DECLINE, STEEPEST_DECLINE = historical_decline(2), historical_decline(5)
if not 0 <= LIKELY_DECLINE <= STEEPEST_DECLINE < 1 or STEEPEST_DECLINE == 0:
    raise ValueError("The triangular liquid-decline assumptions need 0 <= mode <= upper < 1 and upper > 0.")

# The forecast is integrated once for every selected case on a grid of liquid declines; the draws interpolate between grid points
DECLINE_GRID = np.linspace(0, STEEPEST_DECLINE, 13)
oil_grid = np.array([[optimized_new_oil(realization, decline) for decline in DECLINE_GRID] for realization in optimized_cases.index])  # cases x declines x trends x dates
oil_grid = np.concatenate([oil_grid, oil_grid.mean(axis=0, keepdims=True)])                              # last row: average of the cases
liquid_grid = np.array([np.cumsum(declining_liquid(decline) * FORECAST_DAYS) for decline in DECLINE_GRID])
AVERAGE_CASE = len(optimized_cases)

# Calibration uses only out-of-fold training errors; validation and test results remain assessments.
method_errors = hindcast_results.query("partition == 'train_cv' and liquid == 'actual liquid rate' and method == 'average'")['error [%]'].to_numpy() / 100
if not np.isfinite(method_errors).all() or (method_errors <= -1).any():
    raise ValueError("Invalid hindcast errors for observed/predicted correction.")


def observed_to_predicted_factor(relative_error):
    """Invert signed (prediction/observation - 1) errors into multiplicative correction factors."""
    error = np.asarray(relative_error, dtype=float)
    if not np.isfinite(error).all() or (error <= -1).any():
        raise ValueError('Relative errors must be finite and greater than -1.')
    return 1 / (1 + error)


def simulate(vary, seed=SEED):
    """Draw N_DRAWS forecasts of the real field. Sources not listed in `vary` stay at their central value."""
    rng = np.random.default_rng(seed)
    case = rng.integers(len(optimized_cases), size=N_DRAWS) if 'matched case' in vary else np.full(N_DRAWS, AVERAGE_CASE)
    decline = rng.triangular(0, LIKELY_DECLINE, STEEPEST_DECLINE, N_DRAWS) if 'liquid rate' in vary else np.full(N_DRAWS, LIKELY_DECLINE)
    own_share = rng.uniform(0, 1, N_DRAWS) if 'WOR trend' in vary else np.full(N_DRAWS, 0.5)
    error = rng.choice(method_errors, N_DRAWS) if 'method error' in vary else np.zeros(N_DRAWS)

    position = decline / STEEPEST_DECLINE * (len(DECLINE_GRID) - 1)
    below = np.minimum(position.astype(int), len(DECLINE_GRID) - 2)
    weight = (position - below)[:, None]
    oil_by_trend = oil_grid[case, below] * (1 - weight[:, None]) + oil_grid[case, below + 1] * weight[:, None]  # draws x trends x dates
    liquid = liquid_grid[below] * (1 - weight) + liquid_grid[below + 1] * weight
    own, typed = oil_by_trend[:, 0], oil_by_trend[:, 1]
    oil_volume = (own_share[:, None] * own + (1 - own_share[:, None]) * typed) * observed_to_predicted_factor(error)[:, None]
    return field_totals(oil_volume, liquid)


draws = simulate(SOURCES)
percentiles = pd.concat({
    fluid: pd.DataFrame(np.percentile(values, list(PERCENTILES.values()), axis=0).T, index=FORECAST_DATES, columns=list(PERCENTILES))
    for fluid, values in draws.items()
}, axis=1)
percentiles.to_csv(FIGURE_DIR.parent / 'forecast_percentiles.csv')

milestones = percentiles.loc[['2013-01-01', '2018-01-01', '2023-01-01', '2028-01-01']] / 1e6
print(f'Central liquid decline {100 * LIKELY_DECLINE:.2f}% per year, steepest {100 * STEEPEST_DECLINE:.2f}% per year; '
      f'method errors from the hindcast: {100 * method_errors.min():.1f}% to {100 * method_errors.max():.1f}%, mean {100 * method_errors.mean():.1f}%')
for fluid, unit in (('oil', 'million STB'), ('water', 'million STB'), ('gas', 'million MSCF')):
    print(f'\nCumulative {fluid} of the real field [{unit}]:')
    print(milestones[fluid].round(1).to_string())

# Which source matters most? Vary one source at a time and measure the P90-P10 width of 2028 oil
low, high = PERCENTILES['P90 (low case)'], PERCENTILES['P10 (high case)']
width = pd.DataFrame({
    name: dict(zip(('P90', 'P10'), np.percentile(simulate(vary)['oil'][:, -1], [low, high]) / 1e6))
    for name, vary in [(source, [source]) for source in SOURCES] + [('all four together', SOURCES)]
}).T
width['width'] = width['P10'] - width['P90']
print('\nRange of cumulative oil in 2028 [million STB] when only one source of uncertainty is switched on:')
print(width.round(2).to_string())
print(f'\nfor comparison, the {N_SELECTED} forecasts of section 7.3 span {summary["oil 2028 [MMSTB]"].min():.1f} to {summary["oil 2028 [MMSTB]"].max():.1f} million STB')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, fluid in zip(axes, MODELLED_FLUIDS):
    bands = percentiles[fluid] / 1e6
    ax.plot(obs_cum.loc[PLOT_FROM:, fluid] / 1e6, color=COLOURS['observed'], linewidth=2.5)
    ax.fill_between(bands.index, bands['P90 (low case)'], bands['P10 (high case)'], color=COLOURS[fluid], alpha=0.25, linewidth=0)
    ax.plot(bands['P50 (middle case)'], color=COLOURS[fluid], linewidth=2)
    ax.axvline(dates[-1], color=COLOURS['muted'], linestyle='--', linewidth=1)
    last = bands.iloc[-1]
    ax.text(0.03, 0.95, f'2028, P90 / P50 / P10:\n{last.iloc[0]:.0f} / {last.iloc[1]:.0f} / {last.iloc[2]:.0f} million STB', transform=ax.transAxes, va='top')
    ax.set_title(f'Cumulative {fluid} [million STB]')
axes[0].plot([], [], color=COLOURS['observed'], linewidth=2.5, label='Real field up to January 2008')
axes[0].plot([], [], color=COLOURS['neutral'], linewidth=2, label='P50: middle case')
axes[0].fill_between([], [], [], color=COLOURS['neutral'], alpha=0.25, linewidth=0, label='P90 to P10: 80% of the draws fall inside')
axes[0].plot([], [], color=COLOURS['muted'], linestyle='--', label='End of history')
oil_2028 = percentiles['oil'].iloc[-1] / 1e6
finish(fig, f'Conditional 2028 oil: median {oil_2028.iloc[1]:.0f} million STB of oil, '
            f'with an assumed-scenario 80% range of {oil_2028.iloc[0]:.0f} to {oil_2028.iloc[2]:.0f} million STB', 'forecast_percentiles')

### 7.6 Official optimized field forecasts

In [ ]:
# 7.6 Official flat-liquid field forecasts: the same objects are exported in section 8
official_liquid_volume = np.cumsum(declining_liquid(0.0) * FORECAST_DAYS)
optimized_deliverable = {
    realization: pd.DataFrame(field_totals(optimized_new_oil(realization, 0.0).mean(axis=0), official_liquid_volume),
                              index=FORECAST_DATES)
    for realization in optimized_cases.index
}
optimized_summary = pd.DataFrame({
    realization: {'proxy mismatch [%]': optimized_cases.loc[realization, 'mismatch'],
                  'oil 2018 [MMSTB]': forecast.loc['2018-01-01', 'oil'] / 1e6,
                  'oil 2028 [MMSTB]': forecast['oil'].iloc[-1] / 1e6,
                  'water 2028 [MMSTB]': forecast['water'].iloc[-1] / 1e6,
                  'oil rate 2028 [STB/d]': forecast['oil'].diff().iloc[-1] / FORECAST_DAYS[-1]}
    for realization, forecast in optimized_deliverable.items()
}).T.rename_axis('realization_id')
print(optimized_summary.round(3).to_string())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, fluid in zip(axes, ['gas', 'oil', 'water']):
    for forecast in optimized_deliverable.values():
        plotted = pd.concat([obs_cum.loc[dates[-1]:, fluid], forecast[fluid]])
        ax.plot(plotted / 1e6, color=COLOURS[fluid], linewidth=1)
    ax.plot(obs_cum.loc[PLOT_FROM:, fluid] / 1e6, color=COLOURS['observed'], linewidth=2)
    ax.axvline(dates[-1], color=COLOURS['muted'], linestyle='--', linewidth=1)
    unit = 'million MSCF' if fluid == 'gas' else 'million STB'
    ax.set_title(f'Cumulative {fluid} [{unit}]')
axes[0].plot([], [], color=COLOURS['observed'], label='Observed field')
for fluid in ['gas', 'oil', 'water']:
    axes[0].plot([], [], color=COLOURS[fluid], label=f'Ten official {fluid} forecasts')
finish(fig, 'Official workbook forecasts: ten optimized realizations anchored to the observed field',
       'forecast_optimized_cases')

## 8. Deliverable

The official workbook keeps exactly ten sheets, `Case 1`–`Case 10`, and the template's
gas [MSCF], oil [STB], and water [STB] columns. Every sheet contains 80 quarterly dates
from April 2008 to January 2028, including the extension of the shortened first sheet.
Sheet k receives optimized realization k, ranked by the proxy history-match objective.
The base-case liquid rate is flat; every forecast starts at the observed field totals
and is anchored to the last observed WOR. No observed-history correction is presented
as a new reservoir simulator run.

`selected_cases.csv` maps sheets to `realization_id`, all four parameters, proxy mismatch,
search provenance, and forecast milestones. `simulation_case` is empty for new realizations;
`baseline_selected_cases.csv` retains real simulator case numbers separately.
`production_history_and_forecast.csv` supplies observed history, separately labelled
optimized proxy histories, official forecasts with an explicit observed January 2008
anchor, and conditional marginal percentiles. Proxy history series are not spliced into
anchored forecast series: doing so would distort the first forecast-quarter increment.
Official forecast rows are verified against the workbook. Use the official workbook as
the primary Power BI source; the CSV supplies supplementary histories and uncertainty data.

Original simulator benchmark data stays separate from optimized submission identities.
The repository contains the current code, model and numeric outputs. PowerPoint, Power BI
and video are separate deliverables.

In [ ]:
# Export optimized results; retain a separate benchmark for existing report helpers
import copy
import json
import openpyxl

TEMPLATE_FILE = DATA_DIR / '09 Template Deliverable.xlsx'
TEMPLATE_FLUIDS = ['gas', 'oil', 'water']
flat_liquid_volume = np.cumsum(declining_liquid(0.0) * FORECAST_DAYS)

# Original simulator benchmarks stay separate from optimized submission identities.
deliverable = {case: pd.DataFrame(field_totals(new_oil(case, 0.0).mean(axis=0), flat_liquid_volume), index=FORECAST_DATES)
               for case in selected_cases}
baseline_mapping = ranking.loc[selected_cases].reset_index()
baseline_mapping.to_csv(OUTPUT_DIR / 'baseline_selected_cases.csv', index=False)

workbook = openpyxl.load_workbook(TEMPLATE_FILE)
expected_sheets = [f'Case {i}' for i in range(1, N_SELECTED + 1)]
assert workbook.sheetnames == expected_sheets, 'Unexpected official template sheet names or order.'
expected_headers = ['Date', 'Gas production cumulative [MSCF]', 'Oil production cumulative [STB]',
                    'Water production cumulative [STB]']
for sheet, realization in zip(workbook.worksheets, optimized_cases.index):
    assert [sheet.cell(1, col).value for col in range(1, 5)] == expected_headers
    values = optimized_deliverable[realization][TEMPLATE_FLUIDS]
    for row, (date, volumes) in enumerate(values.iterrows(), start=2):
        template_date = sheet.cell(row, 1).value
        if template_date is None:
            for col in range(1, 5):
                sheet.cell(row, col)._style = copy.copy(sheet.cell(row - 1, col)._style)
            sheet.cell(row, 1, date.strftime('%m/%d/%Y'))
        else:
            assert pd.to_datetime(template_date) == date, f'{sheet.title} row {row}: unexpected date.'
        for col, volume in enumerate(volumes, start=2):
            sheet.cell(row, col, float(volume))
output_workbook = OUTPUT_DIR / '09 Template Deliverable.xlsx'
workbook.save(output_workbook)
workbook.close()

saved = openpyxl.load_workbook(output_workbook, read_only=True)
try:
    assert saved.sheetnames == expected_sheets
    for sheet, realization in zip(saved.worksheets, optimized_cases.index):
        rows = list(sheet.iter_rows(min_row=2, max_row=1 + len(FORECAST_DATES), max_col=4, values_only=True))
        assert pd.DatetimeIndex(pd.to_datetime([row[0] for row in rows])).equals(FORECAST_DATES)
        written = np.asarray([row[1:4] for row in rows], dtype=float)
        assert np.allclose(written, optimized_deliverable[realization][TEMPLATE_FLUIDS].to_numpy(), rtol=1e-12, atol=1e-6)

finally:
    saved.close()

optimized_mapping = optimized_cases.reset_index()
optimized_mapping.insert(0, 'sheet', expected_sheets)
optimized_mapping['simulation_case'] = pd.array([pd.NA] * N_SELECTED, dtype='Int64')
for fluid in ('oil', 'water'):
    optimized_mapping[f'{fluid} 2028 [MMSTB]'] = [optimized_deliverable[r][fluid].iloc[-1] / 1e6 for r in optimized_cases.index]
optimized_mapping.to_csv(OUTPUT_DIR / 'selected_cases.csv', index=False)
# Keep the old report helper mapping explicitly on the simulator benchmark.
mapping = baseline_mapping.copy()
mapping.insert(0, 'sheet', expected_sheets)
for fluid in ('oil', 'water'):
    mapping[f'{fluid} 2028 [MMSTB]'] = [deliverable[c][fluid].iloc[-1] / 1e6 for c in selected_cases]

tables = [obs_cum.assign(series='Observed', period='history', source='observed')]
for sheet_name, realization in zip(expected_sheets, optimized_cases.index):
    history = optimized_cum.xs(realization, axis=1, level=1)[TEMPLATE_FLUIDS].assign(period='history')
    future = optimized_deliverable[realization][TEMPLATE_FLUIDS].assign(period='forecast')
    tables.append(history.assign(series=f'{sheet_name} proxy history', realization_id=realization,
                                 source='proxy_prediction', simulation_case=pd.NA))
    anchor = obs_cum.loc[[dates[-1]], TEMPLATE_FLUIDS].assign(period='anchor')
    tables.append(pd.concat([anchor, future]).assign(series=sheet_name, realization_id=realization,
                                                    source='anchored_field_forecast', simulation_case=pd.NA))
for label in PERCENTILES:
    tables.append(percentiles.xs(label, axis=1, level=1)[TEMPLATE_FLUIDS].assign(
        series=label, period='forecast', source='conditional_marginal_percentile'))
powerbi_table = pd.concat(tables).rename_axis('date').reset_index()
powerbi_table = powerbi_table.rename(columns={fluid: f'{fluid}_cumulative' for fluid in TEMPLATE_FLUIDS})
powerbi_table['simulation_case'] = pd.array([pd.NA] * len(powerbi_table), dtype='Int64')
powerbi_table.to_csv(OUTPUT_DIR / 'production_history_and_forecast.csv', index=False)

run_metadata = {'seed': SEED, 'package_versions': PACKAGE_VERSIONS, 'model_fit_cases': fit_cases.to_list(),
                'forecast_start': str(FORECAST_DATES[0].date()), 'forecast_end': str(FORECAST_DATES[-1].date()),
                'forecast_quarters': len(FORECAST_DATES), 'realizations': optimized_cases.index.to_list(),
                'oil_pore_volume_coefficient': OIL_PV_COEFFICIENT,
                'calibration_partition': 'train_cv', 'uncertainty': 'conditional scenario percentiles, not validated twenty-year coverage',
                'evaluation_status': 'reserved-case assessment; historical test exposure disclosed in methodology'}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print(optimized_mapping[['sheet', 'realization_id', 'mismatch', 'oil 2028 [MMSTB]', 'water 2028 [MMSTB]']].round(3).to_string(index=False))
print(f'All ten official sheets read back successfully: {len(FORECAST_DATES)} dates and three fluids each.')
print(f'Official outputs written to {OUTPUT_DIR}. See README.md and docs/METHODOLOGY.md for provenance and limitations.')

## 9. Verification and business interpretation

In [ ]:
# 9. Verification: numerical regressions, training-only exploration and data-flow checks
assert eda_params.index.equals(CASES['train'])
assert set(eda_cum.columns.get_level_values('case')) == set(CASES['train'])
assert eda_grid_features.index.equals(CASES['train'])
assert len(FORECAST_DATES) == 80 and FORECAST_DATES[-1] == pd.Timestamp('2028-01-01')
assert len(optimized_cases) == 10 and optimized_cases.index.is_unique
assert not ((optimized_cases[PARAMS] - lower) / (upper - lower)).round(8).duplicated().any()
assert optimized_cases[PARAMS].ge(lower - 1e-12).all().all() and optimized_cases[PARAMS].le(upper + 1e-12).all().all()
pd.testing.assert_frame_equal(optimized_cases, select_realizations(candidate_parameters, candidate_scores, candidate_sources, N_SELECTED))
for record in hindcast_results.itertuples(index=False):
    fitted_ids = {int(case) for case in record.fit_cases.split(',')}
    assert record.case not in fitted_ids and pd.Timestamp(record.fit_end) <= pd.Timestamp(record.cut)
    assert not fitted_ids.intersection(CASES['test'])
    if record.partition == 'train_cv':
        assert fitted_ids.issubset(set(CASES['train']))
assert len(method_errors) == len(CASES['train']) * len(HINDCAST_CUTS)

# Positive forecast error must reduce a forecast when converted to observed/predicted.
assert np.isclose(120.0 * observed_to_predicted_factor(0.2), 100.0)
try:
    observed_to_predicted_factor(-1.0)
except ValueError:
    pass
else:
    raise AssertionError('Non-invertible forecast errors must be rejected.')

# Perturb held-out cases and all future rows: the pre-cut fitted type curve must not change.
cut = dates.get_loc(pd.Timestamp('2005-01-01'))
reference_curve = build_type_curve(cut, CASES['train'])
original_cum = cum
try:
    cum = cum.copy()
    cum.iloc[cut + 1:] *= 3
    held_out_columns = cum.columns.get_level_values('case').isin(CASES['test'])
    cum.loc[:, held_out_columns] *= 2
    perturbed_curve = build_type_curve(cut, CASES['train'])
    assert all(np.allclose(before, after) for before, after in zip(reference_curve, perturbed_curve))
finally:
    cum = original_cum
try:
    hindcast('2005-01-01', [1], [1], 'invalid_overlap')
except ValueError:
    pass
else:
    raise AssertionError('Hindcast fit/evaluation overlap must be rejected.')
synthetic = field_totals(np.array([8., 20., 20.]), np.array([10., 20., 30.]))
assert np.allclose(synthetic['oil'] - FIELD_START['oil'], [8., 18., 18.])
assert np.allclose(synthetic['water'] - FIELD_START['water'], [2., 2., 12.])
trend = anchor_trend(lambda cumulative: 0.01 * cumulative, 100.0, 80.0, np.log(2.0))
assert np.isclose(trend(80.0), np.log(2.0))
mount_calls = []
colab_root = resolve_project_root(in_colab=True, mount_drive=mount_calls.append)
assert colab_root.as_posix() == '/content/drive/MyDrive/DataWork' and mount_calls == ['/content/drive']
mount_calls.clear()
assert resolve_project_root(str(PROJECT_ROOT), in_colab=True, mount_drive=mount_calls.append) == PROJECT_ROOT
assert not mount_calls
assert resolve_project_root(cwd=PROJECT_ROOT / 'outputs') == PROJECT_ROOT

for realization, values in optimized_deliverable.items():
    assert np.isfinite(values.to_numpy()).all()
    increments = np.diff(np.vstack([FIELD_START[values.columns].to_numpy(), values.to_numpy()]), axis=0)
    assert (increments >= -1e-8).all()
    assert np.allclose(increments[:, values.columns.get_loc('oil')] + increments[:, values.columns.get_loc('water')],
                       declining_liquid(0.0) * FORECAST_DAYS, rtol=1e-10, atol=1e-6)
    csv_rows = powerbi_table.query("realization_id == @realization and period == 'forecast'")
    assert pd.DatetimeIndex(csv_rows['date']).equals(FORECAST_DATES)
    assert np.allclose(csv_rows[[f'{fluid}_cumulative' for fluid in TEMPLATE_FLUIDS]], values[TEMPLATE_FLUIDS])
    csv_path = powerbi_table.query("realization_id == @realization and source == 'anchored_field_forecast'")
    assert len(csv_path) == 81 and csv_path['period'].iloc[0] == 'anchor'
    assert csv_path['date'].iloc[0] == dates[-1]
    assert np.allclose(csv_path[[f'{fluid}_cumulative' for fluid in TEMPLATE_FLUIDS]].iloc[0], FIELD_START[TEMPLATE_FLUIDS])
    csv_increments = csv_path[[f'{fluid}_cumulative' for fluid in TEMPLATE_FLUIDS]].diff().iloc[1:].to_numpy()
    assert np.allclose(csv_increments, increments[:, [values.columns.get_loc(fluid) for fluid in TEMPLATE_FLUIDS]])
for fluid, values in draws.items():
    assert np.isfinite(values).all()
    assert (np.diff(values, axis=-1, prepend=np.full_like(values[..., :1], FIELD_START[fluid])) >= -1e-8).all()
    bands = percentiles[fluid]
    assert (bands['P90 (low case)'] <= bands['P50 (middle case)']).all()
    assert (bands['P50 (middle case)'] <= bands['P10 (high case)']).all()

# Reload in this fresh kernel after defining CurveProxy, as documented by the saved artifact.
reloaded = joblib.load(MODEL_DIR / 'curve_proxy.joblib')
assert reloaded['package_versions']['scikit-learn'] == '1.6.1'
assert set(reloaded['fit_cases']).isdisjoint(CASES['test'])
for fluid in MODELLED_FLUIDS:
    assert np.allclose(reloaded['proxy'][fluid].predict(optimized_cases[PARAMS]), proxy[fluid].predict(optimized_cases[PARAMS]))

verification_results = {'status': 'passed', 'optimized_realizations': 10, 'quarters_per_sheet': 80,
                        'hindcast_rows': len(hindcast_results), 'training_calibration_errors': len(method_errors),
                        'colab_configuration': 'mock mount tested; no live Colab session',
                        'checks': ['training-only response EDA', 'deterministic selection', 'parameter bounds', 'case/time separation',
                                   'error correction', 'quarterly conservation', 'percentile ordering',
                                   'Excel round trip', 'dashboard consistency', 'model reload']}
(OUTPUT_DIR / 'verification_results.json').write_text(json.dumps(verification_results, indent=2), encoding='utf-8')
print('PASS:', verification_results)

In [ ]:
# Production quantities for business interpretation; no invented prices or profitability claim
official_mean = sum(optimized_deliverable.values()) / len(optimized_deliverable)
business_summary = pd.DataFrame({
    'January 2008 observed': FIELD_START,
    'January 2028 official base-case mean': official_mean.iloc[-1],
    '2008–2028 incremental base-case volume': official_mean.iloc[-1] - FIELD_START,
})
print('Volumes: oil/water in STB; gas in MSCF.')
print(business_summary.round(0).to_string())
final_steps = official_mean.diff().iloc[-1]
final_water_cut = final_steps['water'] / (final_steps['oil'] + final_steps['water'])
print(f'Base-case water cut by 2028: {final_water_cut:.1%}; oil rate {final_steps["oil"] / FORECAST_DAYS[-1]:,.0f} STB/day.')
print('Business interpretation: declining oil and increasing water share affect saleable volumes and water-handling requirements.')
print('Liquid-rate scenarios and conditional uncertainty quantify production sensitivity, not investment profitability.')
print('Prices, operating costs, capital spending and operating changes require separate, justified inputs.')
business_summary.to_csv(OUTPUT_DIR / 'business_production_summary.csv')

## 10. Kick Off requirement and submission checklist

| PDF reference | Requirement | Notebook evidence / remaining work |
|---|---|---|
| p.26 | Map parameters to production responses | Section 5: PCA + Gaussian-process oil/water proxy; gas derived with stated units |
| p.26 | Minimize observed-history mismatch | Section 6.2: bounded search and deterministic ranking; section 6.6: residuals before anchoring |
| pp.26,31 | History-matched model and optimized curves | Section 6.6: provided versus optimized histories; residual mismatch and proxy status remain explicit |
| pp.26,30 | Forecast twenty years; static rock properties | Sections 7–8: 80 quarters through January 2028; static parameters and operating assumptions |
| p.28 | 70 train / 15 validation / 15 test cases | Sections 5 and 7.2: split preserved; fitting/calibration excludes test responses |
| p.31 | Pick ten forecast cases | Ten ranked optimized realizations, official workbook, and explicit provenance mapping |
| pp.30,35 | Business understanding and insights | Section 9: incremental oil, water handling, sensitivity, and limits; no unsupported profitability claim |
| p.35 | Accuracy, technical understanding, storytelling | Held-out proxy/hindcast results, method diagram, residuals, uncertainty assumptions, and executable checks |
| p.34 | Code repository: **16 October 2026** | Repository contains notebook, dependencies, models and numeric results; organizer website/archive submission is separate |
| p.34 | PowerPoint documentation: **20 October 2026** | Separate PowerPoint deliverable remains to be prepared |
| p.34 | Power BI visualization: **20 October 2026** | Workbook is the primary source; supplementary CSV available; dashboard remains to be prepared |
| p.34 | Video: **20 October 2026** | Separate deliverable remains to be prepared and submitted |
| p.35 | Collaboration bonus | Document actual team contributions separately; none are invented here |

Use the official challenge instructions for packaging/dashboard/video requirements; supplied starter materials are not redistributed in Git.
The PDF's earlier code deadline takes precedence over the general 20 October timeline.
Running the notebook does not upload submissions. See README.md and docs/METHODOLOGY.md for evaluation provenance, AI assistance and remaining deliverables.